In [41]:
# Author : Raghav Gupta
# ============================================================
# Cell 1: Imports, data preparation check, and configuration
# ============================================================
# This cell imports required libraries, checks whether the prepared
# StackSats BTC analytics dataset exists, prepares it if missing,
# and defines all strategy settings.

import polars as pl
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import subprocess
from itertools import product

try:
    from stacksats.runner.core import StrategyRunner, BacktestConfig
except ImportError:
    from stacksats.runner.core import StrategyRunner
    from stacksats.strategy_types import BacktestConfig

from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.mvrv.core import MVRVStrategy
from stacksats.strategies.stable.signals.momentum import MomentumStrategy
from stacksats.strategies.stable.baselines.uniform import UniformStrategy


# ============================================================
# StackSats prepared dataset check
# ============================================================
# This block prepares the file if it is missing.

btc_path = Path.home() / ".stacksats" / "data" / "bitcoin_analytics.parquet"


raw_brk_path = Path("../data/raw/brk_metrics.parquet")

if not btc_path.exists():
    print(f"Prepared dataset not found at: {btc_path}")
    print("Preparing StackSats analytics dataset...")

    if not raw_brk_path.exists():
        raise FileNotFoundError(
            f"Raw BRK metrics file not found at: {raw_brk_path}. "
            "Please update raw_brk_path to the correct location of brk_metrics.parquet."
        )

    subprocess.run(
        [
            "stacksats",
            "data",
            "prepare",
            "--source",
            str(raw_brk_path),
        ],
        check=True,
    )

if not btc_path.exists():
    raise FileNotFoundError(
        f"Failed to create prepared dataset at {btc_path}."
    )

print(f"Using prepared dataset: {btc_path}")


# ============================================================
# Strategy configuration
# ============================================================

# Budget used per calendar-year evaluation window.
TOTAL_BUDGET_USD = 1000.0

# Train and test periods.
TRAIN_START = "2018-01-01"
TRAIN_END = "2023-12-31"
TEST_START = "2024-01-01"
TEST_END = "2025-12-31"

# Minimum number of rows required for a valid calendar-year evaluation window.
# Normal years have 365 rows; leap years have 366 rows and are kept fully.
WINDOW_SIZE = 365

# Default lookbacks used before grid search selects the best combination.
# These values are overwritten later by the best-performing grid result.
MOMENTUM_LOOKBACK = 45
SMA_LOOKBACK = 180
REGIME_LOOKBACK = 180

# Grid values to test.
MOMENTUM_LOOKBACK_GRID = [30, 45, 60, 90]
SMA_LOOKBACK_GRID = [60, 90, 120, 200]
REGIME_LOOKBACK_GRID = [60, 90, 120, 200]

# Unique lookback values used to create rolling features.
LOOKBACK_DAYS = sorted({
    MOMENTUM_LOOKBACK,
    SMA_LOOKBACK,
    REGIME_LOOKBACK,
})

# Small floor to avoid zero or negative allocation signals.
SIGNAL_FLOOR = 1e-8

# Minimum number of days required for a regime to be evaluated.
MIN_REGIME_DAYS = 20

# Tolerance used to decide whether a result is better, worse, or tied.
STATUS_TOLERANCE_PCT = 1e-6

# Fallback strategy options used if a regime appears in test but was not seen in training.
# The grid search will test each fallback option for every lookback combination.
# The SMA fallback is dynamic because the column depends on the selected SMA lookback.
FALLBACK_STRATEGY_GRID = [
    "sma",
    "stacksats_mvrv_weight",
    "stacksats_momentum_weight",
]

def resolve_fallback_strategy(fallback_strategy, sma_lookback=None):
    """Convert a fallback strategy option into the actual weight column name."""
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    if fallback_strategy == "sma":
        return f"sma_{sma_lookback}d_weight"

    return fallback_strategy

def get_fallback_strategy_cols(sma_lookback=None):
    """Return actual fallback strategy weight columns for the selected SMA lookback."""
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK
    return [
        resolve_fallback_strategy(fallback_strategy, sma_lookback)
        for fallback_strategy in FALLBACK_STRATEGY_GRID
    ]

# Default fallback strategy before grid search overwrites it.
FALLBACK_STRATEGY = resolve_fallback_strategy("sma", SMA_LOOKBACK)

# Candidate strategies used in regime selection.
# DCA is benchmark only and is not selected as a candidate here.
def get_candidate_cols(sma_lookback=None):
    """Return candidate strategy columns for the selected SMA lookback."""
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK
    return [
        "stacksats_mvrv_weight",
        "stacksats_momentum_weight",
        f"sma_{sma_lookback}d_weight",
    ]

CANDIDATE_COLS = get_candidate_cols(SMA_LOOKBACK)
FALLBACK_CANDIDATE_COLS = get_fallback_strategy_cols(SMA_LOOKBACK)


Using prepared dataset: C:\Users\ragha\.stacksats\data\bitcoin_analytics.parquet


In [42]:
# ============================================================
# Cell 2: Helper functions
# ============================================================
# This cell defines general helper functions used across the notebook.
# These functions are reused for labeling results, formatting chart text,
# trimming complete windows, normalizing weights, and summarizing results.

def get_status_from_pct_diff(pct_diff, tolerance=STATUS_TOLERANCE_PCT):
    """
    Convert percentage improvement into a simple status label.

    Parameters
    ----------
    pct_diff : float
        Percentage difference of strategy performance versus DCA.
    tolerance : float
        Small threshold used to avoid classifying tiny numerical differences
        as meaningful wins or losses.

    Returns
    -------
    str
        'better' if strategy beats DCA, 'worse' if it underperforms,
        and 'tie' if the difference is within tolerance.
    """

    # If improvement is greater than tolerance, strategy is better.
    if pct_diff > tolerance:
        return "better"

    # If improvement is below negative tolerance, strategy is worse.
    if pct_diff < -tolerance:
        return "worse"

    # Otherwise treat it as no meaningful difference.
    return "tie"


def format_arrow_text(extra_spd, improvement_pct):
    """
    Create chart annotation text for positive or negative SPD improvement.

    Parameters
    ----------
    extra_spd : float
        Extra sats per dollar compared with DCA.
    improvement_pct : float
        Percentage improvement compared with DCA.

    Returns
    -------
    tuple[str, str]
        Formatted text and color name. Positive values use a green upward
        arrow, while negative values use a red downward arrow.
    """

    # Positive result: green upward arrow.
    if extra_spd >= 0:
        return f"▲ +{extra_spd:,.2f} sats/$ ({improvement_pct:+.2f}%)", "green"

    # Negative result: red downward arrow.
    return f"▼ {extra_spd:,.2f} sats/$ ({improvement_pct:+.2f}%)", "red"


def get_calendar_year_windows(
    df: pd.DataFrame,
    split_start: str,
    split_end: str,
    min_days: int = WINDOW_SIZE,
):
    """
    Build one evaluation window per calendar year.

    This is used instead of fixed 365-row chunking so leap years are handled
    correctly. For example, the 2024 test window is 2024-01-01 to 2024-12-31
    with 366 rows, and the next test window starts on 2025-01-01.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with a datetime `date` column.
    split_start : str
        Inclusive split start date, such as TRAIN_START or TEST_START.
    split_end : str
        Inclusive split end date, such as TRAIN_END or TEST_END.
    min_days : int
        Minimum rows required to keep a calendar-year window. The default is
        365, so normal years and leap years are both valid.

    Returns
    -------
    tuple[pd.DataFrame, list[dict], pd.DataFrame, pd.DataFrame]
        Raw split dataframe, list of calendar-year window dictionaries,
        metadata for kept windows, and metadata for skipped years.
    """

    split_start_ts = pd.to_datetime(split_start)
    split_end_ts = pd.to_datetime(split_end)

    raw_split = (
        df[
            (df["date"] >= split_start_ts) &
            (df["date"] <= split_end_ts)
        ]
        .copy()
        .sort_values("date")
        .reset_index(drop=True)
    )

    windows = []
    kept_rows = []
    skipped_rows = []

    for year in range(split_start_ts.year, split_end_ts.year + 1):
        year_start = max(pd.Timestamp(year=year, month=1, day=1), split_start_ts)
        year_end = min(pd.Timestamp(year=year, month=12, day=31), split_end_ts)

        year_df = (
            raw_split[
                (raw_split["date"] >= year_start) &
                (raw_split["date"] <= year_end)
            ]
            .copy()
            .sort_values("date")
            .reset_index(drop=True)
        )

        observed_days = len(year_df)
        expected_days = (year_end - year_start).days + 1

        if observed_days < min_days:
            skipped_rows.append({
                "year": year,
                "start_date": year_start,
                "end_date": year_end,
                "observed_days": observed_days,
                "expected_calendar_days": expected_days,
                "reason": f"less than {min_days} rows",
            })
            continue

        window_number = len(windows) + 1
        windows.append({
            "window": window_number,
            "year": year,
            "start_date": year_df["date"].min(),
            "end_date": year_df["date"].max(),
            "days": observed_days,
            "expected_calendar_days": expected_days,
            "data": year_df,
        })

        kept_rows.append({
            "window": window_number,
            "year": year,
            "start_date": year_df["date"].min(),
            "end_date": year_df["date"].max(),
            "days": observed_days,
            "expected_calendar_days": expected_days,
            "is_leap_window": observed_days == 366,
        })

    window_metadata_df = pd.DataFrame(kept_rows)
    skipped_metadata_df = pd.DataFrame(skipped_rows)

    return raw_split, windows, window_metadata_df, skipped_metadata_df


def concat_calendar_windows(windows):
    """Concatenate the kept calendar-year windows into one dataframe."""

    if not windows:
        return pd.DataFrame()

    return pd.concat(
        [w["data"].copy() for w in windows],
        ignore_index=True,
    )

def build_simple_normalized_weights(signal_multiplier, signal_floor=SIGNAL_FLOOR):
    """
    Convert signal multipliers into normalized daily allocation weights.

    Parameters
    ----------
    signal_multiplier : array-like
        Raw signal strength or multiplier values.
    signal_floor : float
        Minimum allowed signal value to prevent zero or negative weights.

    Returns
    -------
    np.ndarray
        Daily allocation weights that sum to 1.
    """

    # Convert input into a numpy array.
    signal_multiplier = np.asarray(signal_multiplier, dtype=float)

    # Prevent empty signal arrays.
    if len(signal_multiplier) == 0:
        raise ValueError("Empty signal array.")

    # Replace NaN and infinity values with 0.
    clean_signal = np.nan_to_num(
        signal_multiplier,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    # Apply a small floor so no value is exactly zero or negative.
    clean_signal = np.maximum(clean_signal, signal_floor)

    # If all signals are invalid or zero, fall back to uniform weights.
    if clean_signal.sum() <= 0:
        return np.full(len(clean_signal), 1.0 / len(clean_signal))

    # Normalize so all daily weights sum to 1.
    return clean_signal / clean_signal.sum()


def summarize_spd_like_composite(window_summary_df):
    """
    Summarize performance across all 365-day windows.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary dataframe containing strategy_sats, dca_sats,
        strategy_spd, dca_spd, and result columns.

    Returns
    -------
    dict
        Summary metrics including total sats, SPD sums, improvement percentage,
        wins, losses, ties, and win rate.
    """

    # Number of full 365-day windows.
    n_windows = len(window_summary_df)

    # Sum strategy and DCA sats-per-dollar across windows.
    strategy_spd_sum = window_summary_df["strategy_spd"].sum()
    dca_spd_sum = window_summary_df["dca_spd"].sum()

    # Extra sats-per-dollar gained or lost vs DCA.
    extra_spd_sum = strategy_spd_sum - dca_spd_sum

    # Ratio of strategy SPD to DCA SPD.
    spd_ratio = strategy_spd_sum / dca_spd_sum

    # Percentage improvement over DCA.
    improvement_pct = (spd_ratio - 1.0) * 100.0

    # Sum total sats accumulated by strategy and DCA.
    strategy_sats = window_summary_df["strategy_sats"].sum()
    dca_sats = window_summary_df["dca_sats"].sum()

    # Extra sats accumulated vs DCA.
    extra_sats = strategy_sats - dca_sats

    # Count windows where strategy beat, lost to, or tied DCA.
    wins = int((window_summary_df["result"] == "better").sum())
    losses = int((window_summary_df["result"] == "worse").sum())
    ties = int((window_summary_df["result"] == "tie").sum())

    # Window win rate.
    win_rate_pct = wins / n_windows * 100.0 if n_windows > 0 else 0.0

    # Return one summary dictionary.
    return {
        "n_windows": n_windows,
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "win_rate_pct": win_rate_pct,

        "strategy_sats": strategy_sats,
        "dca_sats": dca_sats,
        "extra_sats_vs_dca": extra_sats,

        "strategy_spd_sum": strategy_spd_sum,
        "dca_spd_sum": dca_spd_sum,
        "extra_spd_sum_vs_dca": extra_spd_sum,

        "strategy_spd_avg": strategy_spd_sum / n_windows,
        "dca_spd_avg": dca_spd_sum / n_windows,
        "extra_spd_avg_vs_dca": extra_spd_sum / n_windows,

        "spd_ratio": spd_ratio,
        "improvement_pct": improvement_pct,
    }


def build_log_tick_values_and_text():
    """
    Build custom y-axis tick values and labels for the BTC log-price chart.

    Returns
    -------
    tuple[list[int], list[str]]
        Tick values and corresponding display labels.
    """

    # Actual BTC price values used as tick positions.
    tickvals = [
        3000, 4000, 5000, 6000, 7000, 8000, 9000,
        10000,
        20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000,
        100000,
    ]

    # Labels displayed on the chart.
    ticktext = [
        "3", "4", "5", "6", "7", "8", "9",
        "10k",
        "2", "3", "4", "5", "6", "7", "8", "9",
        "100k",
    ]

    return tickvals, ticktext


In [43]:
# ============================================================
# Cell 3: Load BTC data
# ============================================================
# This cell loads the prepared Bitcoin analytics parquet file.
# It checks that all required columns exist before moving forward.

if not btc_path.exists():
    raise FileNotFoundError(f"Could not find: {btc_path}")

btc_df = (
    pl.read_parquet(btc_path)
    .with_columns(pl.col("date").cast(pl.Datetime))
    .sort("date")
)

required_cols = [
    "date",
    "price_usd",
    "mvrv",
    "realized_cap_growth_rate",
    "market_cap_growth_rate",
]

missing_cols = [col for col in required_cols if col not in btc_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns from bitcoin_analytics.parquet: {missing_cols}")

print("Loaded BTC rows:", btc_df.height)

display(
    btc_df.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date"),
    )
)


Loaded BTC rows: 5689


min_date,max_date
datetime[μs],datetime[μs]
2010-08-16 00:00:00,2026-03-13 00:00:00


In [44]:
# ============================================================
# Cell 4: StackSats strategy export setup
# ============================================================
# This cell sets up the StackSats strategy runner.
# It exports daily weights from built-in StackSats MVRV, Momentum, and Uniform strategies.
#
# Calendar-year export method used here:
# 1. Export one calendar year at a time: Jan 1 to Dec 31.
# 2. Ask StackSats to export that year.
# 3. Keep the latest export window inside that year using max(end_date).
# 4. Use the exported dates as the valid dates for that year.
#
# Leap-year behavior:
# For a leap year such as 2024, the input date range is still
# 2024-01-01 to 2024-12-31. However, if StackSats internally produces
# 365-day export windows, the latest export window may contain 365 rows
# such as 2024-01-02 to 2024-12-31.

runner = StrategyRunner()

stacksats_strategy_objects = {
    "stacksats_mvrv_weight": MVRVStrategy(),
    "stacksats_momentum_weight": MomentumStrategy(),
    "stacksats_uniform_weight": UniformStrategy(),
}

# Cache prevents recomputing weights for the same strategy/window repeatedly.
_export_cache = {}


def export_stacksats_weight_frame_for_window(
    strategy_key: str,
    window_df: pd.DataFrame,
    full_btc_df: pl.DataFrame,
):
    """
    Export StackSats strategy weights for one calendar-year window.

    After StackSats export, it keeps only the rows whose export end_date
    equals the latest end_date in that calendar year.

    Returns
    -------
    pd.DataFrame
        Columns: date, raw_weight
    """

    # Get current window start and end dates.
    window_start = pd.to_datetime(window_df["date"].min()).strftime("%Y-%m-%d")
    window_end = pd.to_datetime(window_df["date"].max()).strftime("%Y-%m-%d")

    # Cache key is based on strategy and date range.
    cache_key = (strategy_key, window_start, window_end, "latest_end_window")

    # If already computed, return cached version.
    if cache_key in _export_cache:
        return _export_cache[cache_key].copy()

    # Create StackSats export config for this calendar window.
    config = ExportConfig(
        range_start=window_start,
        range_end=window_end,
    )

    # Filter BTC data to the current calendar-year window only.
    window_btc_df = (
        full_btc_df
        .filter(
            (pl.col("date") >= pd.to_datetime(window_start)) &
            (pl.col("date") <= pd.to_datetime(window_end)) &
            pl.col("price_usd").is_not_null()
        )
        .sort("date")
    )

    # Stop if no data exists for this window.
    if window_btc_df.is_empty():
        raise ValueError(f"No BTC data available for {window_start} to {window_end}")

    # Select the requested StackSats strategy object.
    strategy = stacksats_strategy_objects[strategy_key]

    # Run StackSats export.
    export_obj = runner.export(
        strategy,
        config,
        btc_df=window_btc_df,
    )

    # Convert export result to dataframe.
    weights = export_obj.to_dataframe()

    # Ensure output is a Polars dataframe.
    if not isinstance(weights, pl.DataFrame):
        weights = pl.from_pandas(weights)

    # Cast date columns into datetime format.
    weights = weights.with_columns([
        pl.col("start_date").cast(pl.Datetime),
        pl.col("end_date").cast(pl.Datetime),
        pl.col("date").cast(pl.Datetime),
    ])

    # Keep only the latest export window
    latest_end = weights.select(pl.col("end_date").max()).item()

    latest_window_weights = (
        weights
        .filter(pl.col("end_date") == latest_end)
        .select(["date", "weight"])
        .sort("date")
        .rename({"weight": "raw_weight"})
        .to_pandas()
    )

    latest_window_weights["date"] = pd.to_datetime(latest_window_weights["date"])

    # Keep only dates inside the requested calendar-year range.
    start_ts = pd.to_datetime(window_start)
    end_ts = pd.to_datetime(window_end)
    latest_window_weights = latest_window_weights[
        (latest_window_weights["date"] >= start_ts) &
        (latest_window_weights["date"] <= end_ts)
    ].copy()

    if latest_window_weights.empty:
        raise ValueError(
            f"StackSats export returned no usable latest-window weights for {strategy_key} "
            f"from {window_start} to {window_end}."
        )

    # If duplicate dates exist, keep the last one after sorting.
    latest_window_weights = (
        latest_window_weights
        .sort_values("date")
        .drop_duplicates(subset=["date"], keep="last")
        .reset_index(drop=True)
    )

    # Store in cache.
    _export_cache[cache_key] = latest_window_weights.copy()

    return latest_window_weights.copy()


In [45]:
# ============================================================
# Cell 5: Feature engineering and simplified regime classification
# ============================================================
# This cell creates reusable functions for:
# 1. engineering rolling lookback features
# 2. assigning the simpler combined regime
# 3. preparing a clean dataframe for any lookback combination

def classify_btc_mvrv_market_cap_regime(
    row,
    momentum_lookback=None,
    sma_lookback=None,
    regime_lookback=None,
):
    """
    Classify each day into a simpler BTC/on-chain regime.

    The regime combines:
    1. BTC trend regime using SMA ratio and returns
    2. MVRV valuation regime
    3. market-cap / realized-cap growth regime

    Returns
    -------
    str
        Combined regime label in the format:
        BTC trend regime | MVRV valuation regime | cap-growth regime
    """

    if momentum_lookback is None:
        momentum_lookback = MOMENTUM_LOOKBACK
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK
    if regime_lookback is None:
        regime_lookback = REGIME_LOOKBACK

    # ------------------------------------------------------------
    # BTC trend features
    # ------------------------------------------------------------
    sma_selected_ratio = row[f"price_{sma_lookback}d_sma_ratio"]
    sma_regime_ratio = row[f"price_{regime_lookback}d_sma_ratio"]
    return_momentum = row[f"btc_return_{momentum_lookback}d"]
    return_sma = row[f"btc_return_{sma_lookback}d"]

    # ------------------------------------------------------------
    # On-chain / market features
    # ------------------------------------------------------------
    mvrv = row["mvrv"]
    realized_growth = row["realized_cap_growth_rate"]
    market_growth = row["market_cap_growth_rate"]

    # ------------------------------------------------------------
    # 1. BTC trend regime without drawdown
    # ------------------------------------------------------------
    if sma_regime_ratio >= 1.05 and return_sma > 0:
        btc_regime = "BTC Bull"

    elif sma_selected_ratio <= 0.95 and return_sma < 0:
        btc_regime = "BTC Bear"

    elif sma_selected_ratio < 1.0 and return_momentum > 0:
        btc_regime = "BTC Recovery"

    else:
        btc_regime = "BTC Neutral"

    # ------------------------------------------------------------
    # 2. MVRV valuation regime
    # ------------------------------------------------------------
    if mvrv < 1.0:
        valuation_regime = "Low MVRV"

    elif mvrv > 2.5:
        valuation_regime = "High MVRV"

    else:
        valuation_regime = "Normal MVRV"

    # ------------------------------------------------------------
    # 3. Market-cap / realized-cap growth regime
    # ------------------------------------------------------------
    if realized_growth > market_growth:
        cap_regime = "Realized Growth Leading"

    else:
        cap_regime = "Market Growth Leading"

    # ------------------------------------------------------------
    # Final combined regime
    # ------------------------------------------------------------
    return btc_regime + " | " + valuation_regime + " | " + cap_regime




def prepare_btc_data_for_lookbacks(
    full_btc_df,
    momentum_lookback,
    sma_lookback,
    regime_lookback,
):
    """
    Create rolling features and simplified regimes for one lookback combination.
    """

    lookback_days = sorted({
        momentum_lookback,
        sma_lookback,
        regime_lookback,
    })

    feature_exprs = []

    for d in lookback_days:
        feature_exprs.extend([
            pl.col("price_usd")
            .rolling_mean(window_size=d, min_samples=max(3, int(d * 0.30)))
            .alias(f"price_{d}d_sma"),

            pl.col("price_usd")
            .pct_change(d)
            .alias(f"btc_return_{d}d"),
        ])

    temp_btc_df = full_btc_df.with_columns(feature_exprs)

    ratio_exprs = []

    for d in lookback_days:
        ratio_exprs.append(
            (pl.col("price_usd") / pl.col(f"price_{d}d_sma"))
            .alias(f"price_{d}d_sma_ratio")
        )

    temp_btc_df = temp_btc_df.with_columns(ratio_exprs)

    temp_btc_data = temp_btc_df.to_pandas()
    temp_btc_data["date"] = pd.to_datetime(temp_btc_data["date"])

    feature_cols = [
        "price_usd",
        "mvrv",
        "realized_cap_growth_rate",
        "market_cap_growth_rate",
    ]

    for d in lookback_days:
        feature_cols.extend([
            f"price_{d}d_sma",
            f"price_{d}d_sma_ratio",
            f"btc_return_{d}d",
        ])

    temp_btc_data = (
        temp_btc_data
        .dropna(subset=feature_cols)
        .sort_values("date")
        .reset_index(drop=True)
    )

    temp_btc_data["combined_regime"] = temp_btc_data.apply(
        lambda row: classify_btc_mvrv_market_cap_regime(
            row,
            momentum_lookback=momentum_lookback,
            sma_lookback=sma_lookback,
            regime_lookback=regime_lookback,
        ),
        axis=1,
    )

    return temp_btc_data


In [46]:
# ============================================================
# Cell 6: Candidate strategy weights
# ============================================================
# This cell creates daily weights for each candidate strategy.
# Candidate strategies are StackSats MVRV, StackSats Momentum, and custom SMA.
#
# Important:
# StackSats strategies keep the latest export window for each calendar year.
# In leap years, that may be 365 rows instead of 366. To keep all strategies
# comparable, this function evaluates DCA, SMA, MVRV, and Momentum on the
# common exported dates for that year.

def create_candidate_strategy_weights_simple(data, sma_lookback=None):
    """
    Create daily allocation weights for all candidate strategies.

    Parameters
    ----------
    data : pd.DataFrame
        One calendar-year window of BTC data with regime and engineered features.
    sma_lookback : int or None
        SMA lookback used for the custom SMA candidate.

    Returns
    -------
    pd.DataFrame
        Original data plus strategy weight columns:
        - dca_weight
        - stacksats_mvrv_weight
        - stacksats_momentum_weight
        - sma_{sma_lookback}d_weight
    """

    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    # Sort data by date and reset index.
    df = data.copy().sort_values("date").reset_index(drop=True)
    df["date"] = pd.to_datetime(df["date"])

    # Stop if the input window is empty.
    if len(df) == 0:
        raise ValueError("No data available.")

    # Export StackSats MVRV latest-window weights for this calendar year.
    mvrv_weights = export_stacksats_weight_frame_for_window(
        strategy_key="stacksats_mvrv_weight",
        window_df=df,
        full_btc_df=btc_df,
    ).rename(columns={"raw_weight": "stacksats_mvrv_raw_weight"})

    # Export StackSats Momentum latest-window weights for this calendar year.
    momentum_weights = export_stacksats_weight_frame_for_window(
        strategy_key="stacksats_momentum_weight",
        window_df=df,
        full_btc_df=btc_df,
    ).rename(columns={"raw_weight": "stacksats_momentum_raw_weight"})

    # Export StackSats UniformStrategy latest-window weights for this calendar year.
    uniform_weights = export_stacksats_weight_frame_for_window(
        strategy_key="stacksats_uniform_weight",
        window_df=df,
        full_btc_df=btc_df,
    ).rename(columns={"raw_weight": "stacksats_uniform_raw_weight"})

    # Keep only the common dates available in all StackSats exports.
    df = (
        df
        .merge(mvrv_weights, on="date", how="inner")
        .merge(momentum_weights, on="date", how="inner")
        .merge(uniform_weights, on="date", how="inner")
        .sort_values("date")
        .reset_index(drop=True)
    )

    # Number of usable days in the current calendar-year export.
    n = len(df)
    if n < WINDOW_SIZE:
        raise ValueError(
            f"Only {n} common StackSats export dates were available; "
            f"expected at least {WINDOW_SIZE}."
        )

    # Uniform DCA benchmark from StackSats UniformStrategy.
    # Normalize the exported raw uniform weights over the same common dates
    df["dca_weight"] = build_simple_normalized_weights(
        df["stacksats_uniform_raw_weight"].values
    )

    # Normalize StackSats raw weights so each candidate sums to 1 over the
    # same evaluated dates.
    df["stacksats_mvrv_weight"] = build_simple_normalized_weights(
        df["stacksats_mvrv_raw_weight"].values
    )

    df["stacksats_momentum_weight"] = build_simple_normalized_weights(
        df["stacksats_momentum_raw_weight"].values
    )

    # Candidate 3: Custom SMA strategy.
    # price/SMA ratio < 1 means BTC price is below the selected SMA.
    # In that case, sma_signal becomes positive and allocation increases.
    sma_signal = (1.0 - df[f"price_{sma_lookback}d_sma_ratio"]).clip(-1, 1)

    # Convert SMA signal into a multiplier.
    # 1.50 controls how strongly the SMA signal changes allocation.
    sma_multiplier = np.maximum(SIGNAL_FLOOR, 1.0 + 1.50 * sma_signal)

    # Normalize SMA multipliers into daily weights that sum to 1 over the
    # same dates used by StackSats exports.
    df[f"sma_{sma_lookback}d_weight"] = build_simple_normalized_weights(
        sma_multiplier.values
    )

    # Keep raw export columns for debugging, but they are not used for strategy selection.
    return df


In [47]:
# ============================================================
# Cell 7: Calendar-year evaluation functions for one lookback combination
# ============================================================
# This cell evaluates candidate strategies by regime, learns the best mapping,
# and applies that mapping to train/test calendar-year windows.
#
# Leap-year handling:
# - Calendar years are created separately for train and test.
# - For example, 2024 is requested as 2024-01-01 to 2024-12-31.
# - Because StackSats may return rolling 365-day export windows, the latest
#   export window inside a leap year may contain 365 rows, such as
#   2024-01-02 to 2024-12-31.
# - The notebook keeps the export rows with the latest end_date

def evaluate_strategies_by_regime_in_365_windows(
    windows,
    total_budget_usd=TOTAL_BUDGET_USD,
    sma_lookback=None,
):
    """
    Evaluate all candidate strategies inside each regime for every calendar-year window.

    Input is a list of calendar-year windows produced by get_calendar_year_windows().
    """

    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    candidate_cols = get_candidate_cols(sma_lookback)
    rows = []

    for window_info in windows:
        window_idx = window_info["window"]
        year = window_info["year"]
        window_data = window_info["data"].copy().reset_index(drop=True)

        df = create_candidate_strategy_weights_simple(
            data=window_data,
            sma_lookback=sma_lookback,
        )

        for regime, regime_df in df.groupby("combined_regime"):
            if len(regime_df) < MIN_REGIME_DAYS:
                continue

            dca_sats = (
                regime_df["dca_weight"]
                * total_budget_usd
                / regime_df["price_usd"]
                * 100_000_000
            ).sum()

            for col in candidate_cols:
                strategy_sats = (
                    regime_df[col]
                    * total_budget_usd
                    / regime_df["price_usd"]
                    * 100_000_000
                ).sum()

                strategy_spd = strategy_sats / total_budget_usd
                dca_spd = dca_sats / total_budget_usd
                extra_sats = strategy_sats - dca_sats
                extra_spd = strategy_spd - dca_spd
                spd_ratio = strategy_spd / dca_spd
                improvement_pct = (spd_ratio - 1.0) * 100.0

                rows.append({
                    "train_window": window_idx,
                    "year": year,
                    "window_start_date": window_info["start_date"],
                    "window_end_date": window_info["end_date"],
                    "window_days": window_info["days"],
                    "combined_regime": regime,
                    "days": len(regime_df),
                    "strategy": col,
                    "strategy_sats": strategy_sats,
                    "dca_sats": dca_sats,
                    "extra_sats_vs_dca": extra_sats,
                    "strategy_spd": strategy_spd,
                    "dca_spd": dca_spd,
                    "extra_spd_vs_dca": extra_spd,
                    "spd_ratio": spd_ratio,
                    "improvement_pct": improvement_pct,
                    "status": get_status_from_pct_diff(improvement_pct),
                })

    return pd.DataFrame(rows)


def learn_best_mapping_by_regime(train_regime_results_df):
    """
    Learn the best candidate strategy for each regime using training data only.
    """

    best_mapping_df = (
        train_regime_results_df
        .groupby(["combined_regime", "strategy"], as_index=False)
        .agg(
            total_days=("days", "sum"),
            mean_improvement_pct=("improvement_pct", "mean"),
            median_improvement_pct=("improvement_pct", "median"),
            mean_extra_spd_vs_dca=("extra_spd_vs_dca", "mean"),
            total_extra_sats_vs_dca=("extra_sats_vs_dca", "sum"),
            windows_seen=("train_window", "nunique"),
        )
    )

    best_mapping_df = (
        best_mapping_df
        .sort_values(
            ["combined_regime", "mean_improvement_pct", "total_days"],
            ascending=[True, False, False],
        )
        .groupby("combined_regime")
        .head(1)
        .reset_index(drop=True)
    )

    best_mapping_df["status"] = best_mapping_df["mean_improvement_pct"].apply(
        get_status_from_pct_diff
    )

    return best_mapping_df


def apply_regime_mapping_to_one_window(
    window_data,
    best_mapping_df,
    total_budget_usd=TOTAL_BUDGET_USD,
    window_number=None,
    year=None,
    sma_lookback=None,
    fallback_strategy=None,
):
    """
    Apply the learned regime mapping to one calendar-year window.
    """

    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    if fallback_strategy is None:
        fallback_strategy = FALLBACK_STRATEGY

    fallback_strategy = resolve_fallback_strategy(
        fallback_strategy=fallback_strategy,
        sma_lookback=sma_lookback,
    )

    df = create_candidate_strategy_weights_simple(
        data=window_data,
        sma_lookback=sma_lookback,
    )

    regime_to_strategy = dict(
        zip(best_mapping_df["combined_regime"], best_mapping_df["strategy"])
    )

    mapped_strategy = df["combined_regime"].map(regime_to_strategy)
    df["used_fallback_strategy"] = mapped_strategy.isna()
    df["fallback_strategy"] = fallback_strategy
    df["selected_strategy"] = mapped_strategy.fillna(fallback_strategy)

    df["raw_selected_weight"] = df.apply(
        lambda row: row[row["selected_strategy"]],
        axis=1,
    )

    raw_weight_sum = df["raw_selected_weight"].sum()
    if raw_weight_sum <= 0 or pd.isna(raw_weight_sum):
        df["final_strategy_weight"] = 1.0 / len(df)
    else:
        df["final_strategy_weight"] = df["raw_selected_weight"] / raw_weight_sum

    df["final_strategy_usd"] = df["final_strategy_weight"] * total_budget_usd
    df["dca_usd"] = df["dca_weight"] * total_budget_usd

    df["btc_accum_strategy"] = df["final_strategy_usd"] / df["price_usd"]
    df["btc_accum_dca"] = df["dca_usd"] / df["price_usd"]

    df["sats_accum_strategy"] = df["btc_accum_strategy"] * 100_000_000
    df["sats_accum_dca"] = df["btc_accum_dca"] * 100_000_000

    df["strategy_spd_daily"] = df["sats_accum_strategy"] / total_budget_usd
    df["dca_spd_daily"] = df["sats_accum_dca"] / total_budget_usd

    if window_number is not None:
        df["window"] = window_number
    if year is not None:
        df["year"] = year

    return df


def apply_regime_mapping_to_window_set(
    windows,
    best_mapping_df,
    total_budget_usd=TOTAL_BUDGET_USD,
    sma_lookback=None,
    fallback_strategy=None,
):
    """
    Apply the learned regime mapping to every calendar-year window.
    """

    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    if fallback_strategy is None:
        fallback_strategy = FALLBACK_STRATEGY

    fallback_strategy = resolve_fallback_strategy(
        fallback_strategy=fallback_strategy,
        sma_lookback=sma_lookback,
    )

    window_dfs = []
    window_summary_rows = []

    for window_info in windows:
        window_strategy_df = apply_regime_mapping_to_one_window(
            window_data=window_info["data"].copy().reset_index(drop=True),
            best_mapping_df=best_mapping_df,
            total_budget_usd=total_budget_usd,
            window_number=window_info["window"],
            year=window_info["year"],
            sma_lookback=sma_lookback,
            fallback_strategy=fallback_strategy,
        )

        strategy_sats = window_strategy_df["sats_accum_strategy"].sum()
        dca_sats = window_strategy_df["sats_accum_dca"].sum()

        strategy_spd = strategy_sats / total_budget_usd
        dca_spd = dca_sats / total_budget_usd

        extra_sats = strategy_sats - dca_sats
        extra_spd = strategy_spd - dca_spd

        spd_ratio = strategy_spd / dca_spd
        improvement_pct = (spd_ratio - 1.0) * 100.0

        window_summary_rows.append({
            "window": window_info["window"],
            "year": window_info["year"],
            "start_date": window_strategy_df["date"].min(),
            "end_date": window_strategy_df["date"].max(),
            "days": len(window_strategy_df),
            "expected_calendar_days": window_info["expected_calendar_days"],
            "is_leap_window": len(window_strategy_df) == 366,
            "budget_usd": total_budget_usd,
            "strategy_sats": strategy_sats,
            "dca_sats": dca_sats,
            "extra_sats_vs_dca": extra_sats,
            "strategy_spd": strategy_spd,
            "dca_spd": dca_spd,
            "extra_spd_vs_dca": extra_spd,
            "spd_ratio": spd_ratio,
            "improvement_pct": improvement_pct,
            "result": get_status_from_pct_diff(improvement_pct),
            "weight_sum": window_strategy_df["final_strategy_weight"].sum(),
            "max_weight": window_strategy_df["final_strategy_weight"].max(),
            "min_weight": window_strategy_df["final_strategy_weight"].min(),
            "days_above_dca_weight": int(
                (window_strategy_df["final_strategy_weight"] > window_strategy_df["dca_weight"]).sum()
            ),
            "fallback_strategy": fallback_strategy,
            "fallback_days": int(window_strategy_df["used_fallback_strategy"].sum()),
        })

        window_dfs.append(window_strategy_df)

    if not window_dfs:
        raise ValueError("No valid calendar-year windows were available.")

    out_df = pd.concat(window_dfs, ignore_index=True)
    summary_df = pd.DataFrame(window_summary_rows)

    return out_df, summary_df


In [48]:
# ============================================================
# Cell 8: Run full calendar-year strategy for one lookback combination
# ============================================================
# This function is used by the grid search and again for the selected best combo.

def run_full_regime_strategy_for_lookbacks(
    momentum_lookback,
    sma_lookback,
    regime_lookback,
    fallback_strategy=None,
    total_budget_usd=TOTAL_BUDGET_USD,
):
    """
    Run the complete simple-regime strategy for one lookback combination.

    The best regime-to-strategy mapping is learned from training data only,
    then applied separately to train and test windows.
    """

    if fallback_strategy is None:
        fallback_strategy = FALLBACK_STRATEGY

    fallback_strategy = resolve_fallback_strategy(
        fallback_strategy=fallback_strategy,
        sma_lookback=sma_lookback,
    )

    # Prepare features and simplified regime labels for this combination.
    local_btc_data = prepare_btc_data_for_lookbacks(
        full_btc_df=btc_df,
        momentum_lookback=momentum_lookback,
        sma_lookback=sma_lookback,
        regime_lookback=regime_lookback,
    )

    # Train/test split using calendar-year windows.
    # This avoids fixed 365-row chunking across leap years.
    raw_train, local_train_windows, train_calendar_windows_df, train_skipped_years_df = get_calendar_year_windows(
        local_btc_data,
        TRAIN_START,
        TRAIN_END,
        min_days=WINDOW_SIZE,
    )

    raw_test, local_test_windows, test_calendar_windows_df, test_skipped_years_df = get_calendar_year_windows(
        local_btc_data,
        TEST_START,
        TEST_END,
        min_days=WINDOW_SIZE,
    )

    local_train_eval_df = concat_calendar_windows(local_train_windows)
    local_test_eval_df = concat_calendar_windows(local_test_windows)

    local_n_train_windows = len(local_train_windows)
    local_n_test_windows = len(local_test_windows)

    if local_n_train_windows == 0 or local_n_test_windows == 0:
        raise ValueError(
            "Not enough data to create train and test calendar-year windows for this lookback combination."
        )

    # Evaluate candidates by regime on training calendar years only.
    local_train_regime_results_df = evaluate_strategies_by_regime_in_365_windows(
        local_train_windows,
        total_budget_usd=total_budget_usd,
        sma_lookback=sma_lookback,
    )

    if local_train_regime_results_df.empty:
        raise ValueError("No regime-level training results were created.")

    # Learn best strategy per regime from training only.
    local_best_mapping_df = learn_best_mapping_by_regime(
        local_train_regime_results_df
    )

    # Apply learned mapping to train and test periods.
    local_train_strategy_df, local_train_window_summary_df = apply_regime_mapping_to_window_set(
        local_train_windows,
        local_best_mapping_df,
        total_budget_usd=total_budget_usd,
        sma_lookback=sma_lookback,
        fallback_strategy=fallback_strategy,
    )

    local_test_strategy_df, local_test_window_summary_df = apply_regime_mapping_to_window_set(
        local_test_windows,
        local_best_mapping_df,
        total_budget_usd=total_budget_usd,
        sma_lookback=sma_lookback,
        fallback_strategy=fallback_strategy,
    )

    # Summarize train and test performance using the same SPD logic as the final chart.
    local_train_spd_summary = summarize_spd_like_composite(
        local_train_window_summary_df
    )
    local_test_spd_summary = summarize_spd_like_composite(
        local_test_window_summary_df
    )

    local_split_summary_df = pd.DataFrame([
        {
            "split": "train",
            "start_date": raw_train["date"].min(),
            "end_date": raw_train["date"].max(),
            "rows_total": len(raw_train),
            "rows_eval": len(local_train_eval_df),
            "windows": local_n_train_windows,
            "calendar_window_start_dates": ", ".join(train_calendar_windows_df["start_date"].dt.strftime("%Y-%m-%d")),
            "calendar_window_end_dates": ", ".join(train_calendar_windows_df["end_date"].dt.strftime("%Y-%m-%d")),
            "skipped_years": ", ".join(train_skipped_years_df["year"].astype(str)) if not train_skipped_years_df.empty else "",
            "budget_rule": "$1,000 per calendar-year training window; StackSats uses latest export window per year",
        },
        {
            "split": "test",
            "start_date": raw_test["date"].min(),
            "end_date": raw_test["date"].max(),
            "rows_total": len(raw_test),
            "rows_eval": len(local_test_eval_df),
            "windows": local_n_test_windows,
            "calendar_window_start_dates": ", ".join(test_calendar_windows_df["start_date"].dt.strftime("%Y-%m-%d")),
            "calendar_window_end_dates": ", ".join(test_calendar_windows_df["end_date"].dt.strftime("%Y-%m-%d")),
            "skipped_years": ", ".join(test_skipped_years_df["year"].astype(str)) if not test_skipped_years_df.empty else "",
            "budget_rule": "$1,000 per calendar-year test window; StackSats uses latest export window per year",
        },
    ])

    return {
        "momentum_lookback": momentum_lookback,
        "sma_lookback": sma_lookback,
        "regime_lookback": regime_lookback,
        "fallback_strategy": fallback_strategy,
        "btc_data": local_btc_data,
        "split_summary_df": local_split_summary_df,
        "train_calendar_windows_df": train_calendar_windows_df,
        "test_calendar_windows_df": test_calendar_windows_df,
        "train_skipped_years_df": train_skipped_years_df,
        "test_skipped_years_df": test_skipped_years_df,
        "train_eval_df": local_train_eval_df,
        "test_eval_df": local_test_eval_df,
        "train_regime_results_df": local_train_regime_results_df,
        "best_mapping_df": local_best_mapping_df,
        "train_strategy_df": local_train_strategy_df,
        "test_strategy_df": local_test_strategy_df,
        "train_window_summary_df": local_train_window_summary_df,
        "test_window_summary_df": local_test_window_summary_df,
        "train_spd_summary": local_train_spd_summary,
        "test_spd_summary": local_test_spd_summary,
    }


In [49]:
# ============================================================
# Cell 9: Lookback + fallback grid search and best-combination selection
# ============================================================
# This cell tests all combinations of MOMENTUM_LOOKBACK_GRID,
# SMA_LOOKBACK_GRID, REGIME_LOOKBACK_GRID, and FALLBACK_STRATEGY_GRID.
#
# Fallback strategy meaning:
# If a combined regime appears in test but was not seen during training,
# the notebook needs a default strategy for those unseen-regime days.
# Instead of hardcoding only SMA, this grid tests multiple fallback choices.
#
# Ranking logic:
# 1. higher test_improvement_pct is better
# 2. if test improvement ties, higher train_improvement_pct is better
# 3. if both tie, higher test_win_rate_pct is better
# 4. if still tied, higher train_win_rate_pct is better
# 5. if still tied, lower total lookback is preferred

lookback_grid_rows = []
lookback_grid_artifacts = {}

for momentum_lb, sma_lb, regime_lb in product(
    MOMENTUM_LOOKBACK_GRID,
    SMA_LOOKBACK_GRID,
    REGIME_LOOKBACK_GRID,
):
    fallback_strategy_cols = get_fallback_strategy_cols(sma_lb)

    for fallback_col in fallback_strategy_cols:
        combo_key = (momentum_lb, sma_lb, regime_lb, fallback_col)

        try:
            result = run_full_regime_strategy_for_lookbacks(
                momentum_lookback=momentum_lb,
                sma_lookback=sma_lb,
                regime_lookback=regime_lb,
                fallback_strategy=fallback_col,
                total_budget_usd=TOTAL_BUDGET_USD,
            )

            lookback_grid_artifacts[combo_key] = result

            train_summary = result["train_spd_summary"]
            test_summary = result["test_spd_summary"]

            train_fallback_days = int(result["train_strategy_df"]["used_fallback_strategy"].sum())
            test_fallback_days = int(result["test_strategy_df"]["used_fallback_strategy"].sum())

            lookback_grid_rows.append({
                "status": "success",
                "momentum_lookback": momentum_lb,
                "sma_lookback": sma_lb,
                "regime_lookback": regime_lb,
                "fallback_strategy": fallback_col,
                "total_lookback": momentum_lb + sma_lb + regime_lb,

                "train_windows": train_summary["n_windows"],
                "test_windows": test_summary["n_windows"],

                "train_improvement_pct": train_summary["improvement_pct"],
                "test_improvement_pct": test_summary["improvement_pct"],
                "train_extra_spd_sum_vs_dca": train_summary["extra_spd_sum_vs_dca"],
                "test_extra_spd_sum_vs_dca": test_summary["extra_spd_sum_vs_dca"],
                "train_win_rate_pct": train_summary["win_rate_pct"],
                "test_win_rate_pct": test_summary["win_rate_pct"],
                "train_strategy_sats": train_summary["strategy_sats"],
                "test_strategy_sats": test_summary["strategy_sats"],
                "train_dca_sats": train_summary["dca_sats"],
                "test_dca_sats": test_summary["dca_sats"],
                "train_fallback_days": train_fallback_days,
                "test_fallback_days": test_fallback_days,
                "error": "",
            })

        except Exception as exc:
            lookback_grid_rows.append({
                "status": "error",
                "momentum_lookback": momentum_lb,
                "sma_lookback": sma_lb,
                "regime_lookback": regime_lb,
                "fallback_strategy": fallback_col,
                "total_lookback": momentum_lb + sma_lb + regime_lb,
                "train_windows": np.nan,
                "test_windows": np.nan,
                "train_improvement_pct": np.nan,
                "test_improvement_pct": np.nan,
                "train_extra_spd_sum_vs_dca": np.nan,
                "test_extra_spd_sum_vs_dca": np.nan,
                "train_win_rate_pct": np.nan,
                "test_win_rate_pct": np.nan,
                "train_strategy_sats": np.nan,
                "test_strategy_sats": np.nan,
                "train_dca_sats": np.nan,
                "test_dca_sats": np.nan,
                "train_fallback_days": np.nan,
                "test_fallback_days": np.nan,
                "error": str(exc)[:300],
            })

lookback_grid_results_df = pd.DataFrame(lookback_grid_rows)

successful_grid_df = lookback_grid_results_df[
    lookback_grid_results_df["status"] == "success"
].copy()

if successful_grid_df.empty:
    print("No successful grid combinations were found. Showing the first 20 error rows:")
    pd.set_option("display.max_colwidth", 300)
    display(
        lookback_grid_results_df[
            [
                "momentum_lookback",
                "sma_lookback",
                "regime_lookback",
                "fallback_strategy",
                "error",
            ]
        ].head(20)
    )
    print("\nMost common error messages:")
    display(
        lookback_grid_results_df["error"]
        .value_counts()
        .head(10)
        .reset_index()
        .rename(columns={"index": "error", "error": "count"})
    )
    raise ValueError("No successful lookback/fallback combinations were found. See the error column above.")

ranked_lookback_grid_df = (
    successful_grid_df
    .sort_values(
        [
            "test_improvement_pct",
            "train_improvement_pct",
            "test_win_rate_pct",
            "train_win_rate_pct",
            "total_lookback",
        ],
        ascending=[False, False, False, False, True],
    )
    .reset_index(drop=True)
)

ranked_lookback_grid_df["rank"] = np.arange(1, len(ranked_lookback_grid_df) + 1)

best_lookback_row = ranked_lookback_grid_df.iloc[0]
best_combo_key = (
    int(best_lookback_row["momentum_lookback"]),
    int(best_lookback_row["sma_lookback"]),
    int(best_lookback_row["regime_lookback"]),
    str(best_lookback_row["fallback_strategy"]),
)

# Overwrite global lookback and fallback settings with selected best values.
MOMENTUM_LOOKBACK = best_combo_key[0]
SMA_LOOKBACK = best_combo_key[1]
REGIME_LOOKBACK = best_combo_key[2]
FALLBACK_STRATEGY = best_combo_key[3]
LOOKBACK_DAYS = sorted({MOMENTUM_LOOKBACK, SMA_LOOKBACK, REGIME_LOOKBACK})
CANDIDATE_COLS = get_candidate_cols(SMA_LOOKBACK)
FALLBACK_CANDIDATE_COLS = get_fallback_strategy_cols(SMA_LOOKBACK)

# Pull selected artifacts into the same variable names used by the final cells.
selected_result = lookback_grid_artifacts[best_combo_key]

btc_data = selected_result["btc_data"]
split_summary_df = selected_result["split_summary_df"]
train_eval_df = selected_result["train_eval_df"]
test_eval_df = selected_result["test_eval_df"]
train_calendar_windows_df = selected_result["train_calendar_windows_df"]
test_calendar_windows_df = selected_result["test_calendar_windows_df"]
train_skipped_years_df = selected_result["train_skipped_years_df"]
test_skipped_years_df = selected_result["test_skipped_years_df"]
train_regime_results_df = selected_result["train_regime_results_df"]
best_mapping_df = selected_result["best_mapping_df"]
train_strategy_df = selected_result["train_strategy_df"]
test_strategy_df = selected_result["test_strategy_df"]
train_window_summary_df = selected_result["train_window_summary_df"]
test_window_summary_df = selected_result["test_window_summary_df"]
train_spd_summary = selected_result["train_spd_summary"]
test_spd_summary = selected_result["test_spd_summary"]

print("Best lookback + fallback combination selected:")
print(f"MOMENTUM_LOOKBACK = {MOMENTUM_LOOKBACK}")
print(f"SMA_LOOKBACK      = {SMA_LOOKBACK}")
print(f"REGIME_LOOKBACK   = {REGIME_LOOKBACK}")
print(f"FALLBACK_STRATEGY = {FALLBACK_STRATEGY}")

print("\nFallback strategy options tested for selected SMA lookback:")
print(FALLBACK_CANDIDATE_COLS)

print("\nTop 20 lookback + fallback combinations ranked by test improvement, then train improvement:")
display(
    ranked_lookback_grid_df[
        [
            "rank",
            "momentum_lookback",
            "sma_lookback",
            "regime_lookback",
            "fallback_strategy",
            "test_improvement_pct",
            "train_improvement_pct",
            "test_extra_spd_sum_vs_dca",
            "train_extra_spd_sum_vs_dca",
            "test_win_rate_pct",
            "train_win_rate_pct",
            "test_fallback_days",
            "train_fallback_days",
            "total_lookback",
        ]
    ]
    .head(20)
    .round(6)
)

print("\nSelected train/test split summary:")
display(split_summary_df)

print("\nSelected regime sample:")
display(
    btc_data[["date", "price_usd", "combined_regime"]]
    .head()
)


Best lookback + fallback combination selected:
MOMENTUM_LOOKBACK = 60
SMA_LOOKBACK      = 200
REGIME_LOOKBACK   = 200
FALLBACK_STRATEGY = stacksats_mvrv_weight

Fallback strategy options tested for selected SMA lookback:
['sma_200d_weight', 'stacksats_mvrv_weight', 'stacksats_momentum_weight']

Top 20 lookback + fallback combinations ranked by test improvement, then train improvement:


,rank,momentum_lookback,sma_lookback,regime_lookback,fallback_strategy,test_improvement_pct,train_improvement_pct,test_extra_spd_sum_vs_dca,train_extra_spd_sum_vs_dca,test_win_rate_pct,train_win_rate_pct,test_fallback_days,train_fallback_days,total_lookback
0,1,60,200,200,stacksats_mvrv_weight,4.135221,1.123569,106.864212,567.545898,100.0,83.333333,15,67,460
1,2,45,200,200,stacksats_mvrv_weight,4.125536,1.076833,106.613928,543.937969,100.0,83.333333,26,64,445
2,3,90,200,200,stacksats_mvrv_weight,4.125536,1.058167,106.613928,534.509530,100.0,83.333333,17,60,490
3,4,60,200,200,stacksats_momentum_weight,4.070304,1.194634,105.186608,603.442745,100.0,83.333333,15,67,460
4,5,60,200,200,sma_200d_weight,4.034617,1.260316,104.264354,636.620433,100.0,83.333333,15,67,460
5,6,90,200,200,stacksats_momentum_weight,4.012938,1.200313,103.704129,606.311125,100.0,83.333333,17,60,490
6,7,45,200,200,stacksats_momentum_weight,3.984711,1.129375,102.974673,570.478619,100.0,83.333333,26,64,445
7,8,30,200,200,sma_200d_weight,3.959208,1.267718,102.315599,640.359176,100.0,83.333333,0,10,430
8,9,30,200,200,stacksats_momentum_weight,3.959208,1.191550,102.315599,601.884758,100.0,83.333333,0,10,430
9,10,30,200,200,stacksats_mvrv_weight,3.959208,1.115591,102.315599,563.515984,100.0,83.333333,0,10,430



Selected train/test split summary:


,split,start_date,end_date,rows_total,rows_eval,windows,calendar_window_start_dates,calendar_window_end_dates,skipped_years,budget_rule
0,train,2018-01-01,2023-12-31,2191,2191,6,"2018-01-01, 2019-01-01, 2020-01-01, 2021-01-01...","2018-12-31, 2019-12-31, 2020-12-31, 2021-12-31...",,"$1,000 per calendar-year training window; Stac..."
1,test,2024-01-01,2025-12-31,731,731,2,"2024-01-01, 2025-01-01","2024-12-31, 2025-12-31",,"$1,000 per calendar-year test window; StackSat..."



Selected regime sample:


,date,price_usd,combined_regime
0,2011-08-16,11.05,BTC Bull | Normal MVRV | Realized Growth Leading
1,2011-08-17,10.88,BTC Bull | Normal MVRV | Realized Growth Leading
2,2011-08-18,10.90,BTC Bull | Normal MVRV | Realized Growth Leading
3,2011-08-19,11.40,BTC Bull | Normal MVRV | Realized Growth Leading
4,2011-08-20,11.49,BTC Bull | Normal MVRV | Realized Growth Leading


In [50]:
# ============================================================
# Final cell: Run StackSats backtest for selected multi-strategy regime approach
# ============================================================
# Run this full cell after the grid-search / selected-strategy cells have finished.
# This version follows the current StackSats strategy contract:
# - Do not override compute_weights().
# - Implement propose_weight(state).
# - Use BaseStrategy metadata fields, not a plain metadata dictionary.

# Required objects from earlier cells:
# - train_strategy_df
# - test_strategy_df
# - btc_df
# - runner
# - BacktestConfig
# - TEST_START
# - TEST_END

required_names = [
    "train_strategy_df",
    "test_strategy_df",
    "btc_df",
    "runner",
    "BacktestConfig",
    "TEST_START",
    "TEST_END",
]

missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Please run the earlier notebook cells first. Missing required object(s): "
        + ", ".join(missing_names)
    )

# Import the StackSats base strategy so we inherit the built-in compute_weights().
# The runner does not allow custom compute_weights() overrides.
try:
    from stacksats.strategy_types import BaseStrategy
except ImportError:
    try:
        from stacksats.strategies.base import BaseStrategy
    except ImportError:
        from stacksats.strategy.base import BaseStrategy

btc_df_pl = btc_df

# Build one daily lookup table with the final selected regime-based weight.
multi_strategy_weight_lookup_df = (
    pd.concat(
        [
            train_strategy_df[["date", "final_strategy_weight", "combined_regime", "selected_strategy", "used_fallback_strategy"]],
            test_strategy_df[["date", "final_strategy_weight", "combined_regime", "selected_strategy", "used_fallback_strategy"]],
        ],
        ignore_index=True,
    )
    .assign(date=lambda df: pd.to_datetime(df["date"]).dt.normalize())
    .sort_values("date")
    .drop_duplicates(subset=["date"], keep="last")
    .rename(columns={"final_strategy_weight": "weight"})
    .reset_index(drop=True)
)

# Keep weights valid and safe.
multi_strategy_weight_lookup_df["weight"] = (
    multi_strategy_weight_lookup_df["weight"]
    .astype(float)
    .clip(lower=0.0)
)


class MultiStrategyRegimeBasedStrategy(BaseStrategy):
    """
    Backtest-ready wrapper for the selected multi-strategy regime approach.

    Important:
    - The regime mapping has already been learned in the earlier cells.
    - This class supplies one weight per date using propose_weight(state).
    - It intentionally does not override compute_weights(), because current
      StackSats versions block custom compute_weights() overrides.
    - It relies on BaseStrategy.metadata(), which returns the StrategyMetadata
      object StackSats expects. Do not return a plain dict from metadata().
    """

    strategy_id = "multi-strategy-regime-based"
    version = "1.0.0"
    description = "Selected multi-strategy regime-based allocation using learned regime mapping and fallback strategy."

    def __init__(self, weight_lookup_df: pd.DataFrame):
        super().__init__()

        lookup_df = (
            weight_lookup_df[["date", "weight"]]
            .copy()
            .assign(date=lambda df: pd.to_datetime(df["date"]).dt.normalize())
            .sort_values("date")
            .drop_duplicates(subset=["date"], keep="last")
            .reset_index(drop=True)
        )

        # Dictionary lookup is simpler and faster inside propose_weight().
        self.weight_by_date = dict(
            zip(
                lookup_df["date"].dt.strftime("%Y-%m-%d"),
                lookup_df["weight"].astype(float),
            )
        )


    def params(self):
        """Keep provenance compact instead of storing the full daily lookup table."""
        return {
            "lookup_days": len(self.weight_by_date),
            "source": "selected_train_test_regime_strategy_weights",
        }

    def _date_from_value(self, value):
        """Convert any date-like value into a YYYY-MM-DD string."""
        if value is None:
            return None

        try:
            # Some state values may be Polars/Pandas scalar wrappers.
            return pd.to_datetime(value).normalize().strftime("%Y-%m-%d")
        except Exception:
            return None

    def _extract_state_date(self, state):
        """
        Extract the current date from the StackSats strategy state.

        This is intentionally flexible because StackSats state objects may be
        plain objects, dataclasses, dictionaries, or row-like containers.
        """
        possible_date_fields = [
            "date",
            "current_date",
            "timestamp",
            "time",
            "as_of_date",
            "end_date",
            "start_date",
        ]

        # Dictionary-like state.
        if isinstance(state, dict):
            for key in possible_date_fields:
                date_key = self._date_from_value(state.get(key))
                if date_key is not None:
                    return date_key

            # Nested row/features dictionary fallback.
            for nested_key in ["row", "features", "data", "btc_row"]:
                nested = state.get(nested_key)
                if isinstance(nested, dict):
                    for key in possible_date_fields:
                        date_key = self._date_from_value(nested.get(key))
                        if date_key is not None:
                            return date_key

        # Object/dataclass-like state.
        for attr in possible_date_fields:
            if hasattr(state, attr):
                date_key = self._date_from_value(getattr(state, attr))
                if date_key is not None:
                    return date_key

        # Nested object fallback.
        for nested_attr in ["row", "features", "data", "btc_row"]:
            if hasattr(state, nested_attr):
                nested = getattr(state, nested_attr)

                if isinstance(nested, dict):
                    for key in possible_date_fields:
                        date_key = self._date_from_value(nested.get(key))
                        if date_key is not None:
                            return date_key

                for attr in possible_date_fields:
                    if hasattr(nested, attr):
                        date_key = self._date_from_value(getattr(nested, attr))
                        if date_key is not None:
                            return date_key

        raise ValueError(
            "Could not extract date from StackSats state object. "
            f"State type received: {type(state)}"
        )

    def propose_weight(self, state):
        """
        StackSats calls this once per date.
        Return the selected regime-based allocation weight for that date.
        """
        date_key = self._extract_state_date(state)
        return float(self.weight_by_date.get(date_key, 0.0))

    def to_weights(self, btc_df_input):
        """Manual helper for checking weights outside StackSats."""
        if isinstance(btc_df_input, pl.DataFrame):
            dates = btc_df_input.select(pl.col("date")).to_pandas()["date"]
        else:
            dates = pd.to_datetime(btc_df_input["date"])

        out = pd.DataFrame({"date": pd.to_datetime(dates).dt.normalize()})
        out["date_key"] = out["date"].dt.strftime("%Y-%m-%d")
        out["weight"] = out["date_key"].map(self.weight_by_date).fillna(0.0).astype(float)
        return pl.from_pandas(out[["date", "weight"]])


# Create the StackSats-compatible strategy object.
multi_strategy_regime = MultiStrategyRegimeBasedStrategy(
    multi_strategy_weight_lookup_df
)

# Optional sanity check before running StackSats backtest.
manual_weight_check_df = multi_strategy_regime.to_weights(btc_df_pl).to_pandas()
print("Manual weight check:")
print(manual_weight_check_df.head())
print(manual_weight_check_df.tail())

# Backtest period. Default: final holdout test period.
start_date = pd.to_datetime(TEST_START).date()
end_date = pd.to_datetime(TEST_END).date()

multi_strategy_bt = runner.backtest(
    multi_strategy_regime,
    config=BacktestConfig(
        start_date=str(start_date),
        end_date=str(end_date),
    ),
    btc_df=btc_df_pl,
)

multi_strategy_backtest_df = multi_strategy_bt.to_dataframe()

print("Multi-strategy regime-based backtest complete")
print(f"Backtest period: {start_date} to {end_date}")

display(multi_strategy_backtest_df)

# Optional direct call, matching your example style:
# multi_strategy_bt.to_dataframe()


Manual weight check:
        date  weight
0 2010-08-16     0.0
1 2010-08-17     0.0
2 2010-08-18     0.0
3 2010-08-19     0.0
4 2010-08-20     0.0
           date  weight
5684 2026-03-09     0.0
5685 2026-03-10     0.0
5686 2026-03-11     0.0
5687 2026-03-12     0.0
5688 2026-03-13     0.0
Multi-strategy regime-based backtest complete
Backtest period: 2024-01-01 to 2025-12-31


window,min_sats_per_dollar,max_sats_per_dollar,uniform_sats_per_dollar,dynamic_sats_per_dollar,uniform_percentile,dynamic_percentile,excess_percentile
str,f64,f64,f64,f64,f64,f64,f64
"""2024-01-01 → 2024-12-30""",942.777195,2528.060843,1589.861093,1681.872788,40.818178,46.622293,5.804116
"""2024-01-02 → 2024-12-31""",942.777195,2528.060843,1586.574307,1681.794051,40.610847,46.617327,6.00648
"""2024-01-03 → 2025-01-01""",942.777195,2528.060843,1583.375013,1678.948475,40.409035,46.437827,6.028793
"""2024-01-04 → 2025-01-02""",942.777195,2528.060843,1579.810682,1675.783059,40.184196,46.238152,6.053956
"""2024-01-05 → 2025-01-03""",942.777195,2528.060843,1576.393441,1672.746876,39.968636,46.046629,6.077993
…,…,…,…,…,…,…,…
"""2024-12-28 → 2025-12-27""",801.416263,1311.019777,996.869049,1008.299875,38.353893,40.596975,2.243082
"""2024-12-29 → 2025-12-28""",801.416263,1311.019777,997.111228,1008.532139,38.401416,40.642553,2.241137
"""2024-12-30 → 2025-12-29""",801.416263,1311.019777,997.331164,1008.739713,38.444574,40.683285,2.238711


In [51]:
# ============================================================
# Cell 10: Selected-combination details
# ============================================================
# This cell shows the selected grid result, candidate win rate by regime,
# learned best strategy mapping, and window-level train/test results.

selected_lookback_summary_df = pd.DataFrame([
    {
        "selected_metric_order": "test_improvement_pct -> train_improvement_pct -> win rates -> lower total_lookback",
        "momentum_lookback": MOMENTUM_LOOKBACK,
        "sma_lookback": SMA_LOOKBACK,
        "regime_lookback": REGIME_LOOKBACK,
        "fallback_strategy": FALLBACK_STRATEGY,
        "train_improvement_pct": train_spd_summary["improvement_pct"],
        "test_improvement_pct": test_spd_summary["improvement_pct"],
        "train_win_rate_pct": train_spd_summary["win_rate_pct"],
        "test_win_rate_pct": test_spd_summary["win_rate_pct"],
    }
])

display(selected_lookback_summary_df.round(6))

candidate_regime_winrate_df = (
    train_regime_results_df
    .groupby("strategy", as_index=False)
    .agg(
        regime_cases=("status", "count"),
        wins=("status", lambda x: (x == "better").sum()),
        losses=("status", lambda x: (x == "worse").sum()),
        ties=("status", lambda x: (x == "tie").sum()),
        avg_improvement_pct=("improvement_pct", "mean"),
    )
)

candidate_regime_winrate_df["win_rate_pct"] = (
    candidate_regime_winrate_df["wins"]
    / candidate_regime_winrate_df["regime_cases"]
    * 100.0
)

print("Candidate win rate by regime on selected training setup:")
display(
    candidate_regime_winrate_df
    .sort_values("win_rate_pct", ascending=False)
    .round(6)
)

print("Best strategy mapping learned from training only:")
display(
    best_mapping_df
    .sort_values("mean_improvement_pct", ascending=False)
    .round(6)
)

print("Train window results for selected combination:")
display(train_window_summary_df.round(6))

print("Test window results for selected combination:")
display(test_window_summary_df.round(6))


,selected_metric_order,momentum_lookback,sma_lookback,regime_lookback,fallback_strategy,train_improvement_pct,test_improvement_pct,train_win_rate_pct,test_win_rate_pct
0,test_improvement_pct -> train_improvement_pct ...,60,200,200,stacksats_mvrv_weight,1.123569,4.135221,83.333333,100.0


Candidate win rate by regime on selected training setup:


,strategy,regime_cases,wins,losses,ties,avg_improvement_pct,win_rate_pct
0,sma_200d_weight,26,16,10,0,8.726821,61.538462
1,stacksats_momentum_weight,26,12,14,0,0.954083,46.153846
2,stacksats_mvrv_weight,26,9,13,4,7.016673,34.615385


Best strategy mapping learned from training only:


,combined_regime,strategy,total_days,mean_improvement_pct,median_improvement_pct,mean_extra_spd_vs_dca,total_extra_sats_vs_dca,windows_seen,status
3,BTC Bull | High MVRV | Market Growth Leading,stacksats_mvrv_weight,198,153.466398,153.466398,581.540503,1.163081e+06,2,better
6,BTC Neutral | Normal MVRV | Market Growth Leading,stacksats_mvrv_weight,255,96.473639,105.331412,1202.624475,4.810498e+06,4,better
1,BTC Bear | Normal MVRV | Market Growth Leading,sma_200d_weight,91,65.353513,65.353513,1144.715126,2.289430e+06,2,better
0,BTC Bear | Low MVRV | Realized Growth Leading,sma_200d_weight,301,40.422102,39.605229,2153.418374,6.460255e+06,3,better
7,BTC Neutral | Normal MVRV | Realized Growth Le...,sma_200d_weight,212,11.791385,4.709291,-3.172437,-1.586218e+04,5,better
2,BTC Bear | Normal MVRV | Realized Growth Leading,sma_200d_weight,305,5.501472,5.501472,379.267061,7.585341e+05,2,better
5,BTC Bull | Normal MVRV | Realized Growth Leading,stacksats_momentum_weight,253,-2.360186,-2.311805,-26.539190,-1.061568e+05,4,worse
4,BTC Bull | Normal MVRV | Market Growth Leading,stacksats_momentum_weight,445,-3.682536,-1.212434,-91.427956,-3.657118e+05,4,worse


...

C:\Users\ragha\AppData\Local\Temp\ipykernel_8320\1998959187.py:56: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(train_window_summary_df.round(6))


,window,year,start_date,end_date,days,expected_calendar_days,is_leap_window,budget_usd,strategy_sats,dca_sats,...,extra_spd_vs_dca,spd_ratio,improvement_pct,result,weight_sum,max_weight,min_weight,days_above_dca_weight,fallback_strategy,fallback_days
0,1,2018,2018-01-01,2018-12-31,365,365,False,1000.0,1.533483e+07,1.473606e+07,...,598.764720,1.040633,4.063261,better,1.0,0.003768,0.000009,158,stacksats_mvrv_weight,17
1,2,2019,2019-01-01,2019-12-31,365,365,False,1000.0,1.681627e+07,1.592086e+07,...,895.408025,1.056241,5.624118,better,1.0,0.030970,0.000366,156,stacksats_mvrv_weight,0
2,3,2020,2020-01-02,2020-12-31,365,366,False,1000.0,8.630183e+06,1.002682e+07,...,-1396.634837,0.860710,-13.928993,worse,1.0,0.068466,0.000022,62,stacksats_mvrv_weight,17
3,4,2021,2021-01-01,2021-12-31,365,365,False,1000.0,2.521118e+06,2.205550e+06,...,315.568064,1.143079,14.307911,better,1.0,0.078543,0.000008,82,stacksats_mvrv_weight,24
4,5,2022,2022-01-01,2022-12-31,365,365,False,1000.0,4.087924e+06,4.018027e+06,...,69.896828,1.017396,1.739581,better,1.0,0.003537,0.002002,178,stacksats_mvrv_weight,7
5,6,2023,2023-01-01,2023-12-31,365,365,False,1000.0,3.689976e+06,3.605433e+06,...,84.543098,1.023449,2.344880,better,1.0,0.013825,0.000200,106,stacksats_mvrv_weight,2


Test window results for selected combination:


C:\Users\ragha\AppData\Local\Temp\ipykernel_8320\1998959187.py:59: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(test_window_summary_df.round(6))


,window,year,start_date,end_date,days,expected_calendar_days,is_leap_window,budget_usd,strategy_sats,dca_sats,...,extra_spd_vs_dca,spd_ratio,improvement_pct,result,weight_sum,max_weight,min_weight,days_above_dca_weight,fallback_strategy,fallback_days
0,1,2024,2024-01-02,2024-12-31,365,366,False,1000.0,1.681740e+06,1.586574e+06,...,95.166111,1.059982,5.998213,better,1.0,0.088989,0.000009,59,stacksats_mvrv_weight,14
1,2,2025,2025-01-01,2025-12-31,365,365,False,1000.0,1.009368e+06,9.976701e+05,...,11.698102,1.011725,1.172542,better,1.0,0.010043,0.000009,138,stacksats_mvrv_weight,1


In [52]:
# ============================================================
# Cell 11: Final summary tables
# ============================================================
# This cell creates final summary tables for train and test.
# It reports total sats, SPD, improvement percentage, and window win rate.

window_winrate_df = pd.DataFrame([
    {
        "period": "train",
        "windows": len(train_window_summary_df),
        "wins": int((train_window_summary_df["result"] == "better").sum()),
        "losses": int((train_window_summary_df["result"] == "worse").sum()),
        "ties": int((train_window_summary_df["result"] == "tie").sum()),
        "win_rate_pct": (
            (train_window_summary_df["result"] == "better").sum()
            / len(train_window_summary_df)
            * 100.0
            if len(train_window_summary_df) > 0 else 0.0
        ),
    },
    {
        "period": "test",
        "windows": len(test_window_summary_df),
        "wins": int((test_window_summary_df["result"] == "better").sum()),
        "losses": int((test_window_summary_df["result"] == "worse").sum()),
        "ties": int((test_window_summary_df["result"] == "tie").sum()),
        "win_rate_pct": (
            (test_window_summary_df["result"] == "better").sum()
            / len(test_window_summary_df)
            * 100.0
            if len(test_window_summary_df) > 0 else 0.0
        ),
    },
])

display(window_winrate_df.round(2))


train_spd_summary = summarize_spd_like_composite(train_window_summary_df)
test_spd_summary = summarize_spd_like_composite(test_window_summary_df)

final_summary_df = pd.DataFrame([
    {
        "period": "train",
        "start_date": TRAIN_START,
        "end_date": TRAIN_END,
        "windows": train_spd_summary["n_windows"],
        "wins": train_spd_summary["wins"],
        "losses": train_spd_summary["losses"],
        "ties": train_spd_summary["ties"],
        "win_rate_pct": train_spd_summary["win_rate_pct"],

        "strategy_sats": train_spd_summary["strategy_sats"],
        "dca_sats": train_spd_summary["dca_sats"],
        "extra_sats_vs_dca": train_spd_summary["extra_sats_vs_dca"],

        "strategy_spd_sum": train_spd_summary["strategy_spd_sum"],
        "dca_spd_sum": train_spd_summary["dca_spd_sum"],
        "extra_spd_sum_vs_dca": train_spd_summary["extra_spd_sum_vs_dca"],

        "strategy_spd_avg": train_spd_summary["strategy_spd_avg"],
        "dca_spd_avg": train_spd_summary["dca_spd_avg"],
        "extra_spd_avg_vs_dca": train_spd_summary["extra_spd_avg_vs_dca"],

        "spd_ratio": train_spd_summary["spd_ratio"],
        "improvement_pct": train_spd_summary["improvement_pct"],
        "result": get_status_from_pct_diff(train_spd_summary["improvement_pct"]),
    },
    {
        "period": "test",
        "start_date": TEST_START,
        "end_date": TEST_END,
        "windows": test_spd_summary["n_windows"],
        "wins": test_spd_summary["wins"],
        "losses": test_spd_summary["losses"],
        "ties": test_spd_summary["ties"],
        "win_rate_pct": test_spd_summary["win_rate_pct"],

        "strategy_sats": test_spd_summary["strategy_sats"],
        "dca_sats": test_spd_summary["dca_sats"],
        "extra_sats_vs_dca": test_spd_summary["extra_sats_vs_dca"],

        "strategy_spd_sum": test_spd_summary["strategy_spd_sum"],
        "dca_spd_sum": test_spd_summary["dca_spd_sum"],
        "extra_spd_sum_vs_dca": test_spd_summary["extra_spd_sum_vs_dca"],

        "strategy_spd_avg": test_spd_summary["strategy_spd_avg"],
        "dca_spd_avg": test_spd_summary["dca_spd_avg"],
        "extra_spd_avg_vs_dca": test_spd_summary["extra_spd_avg_vs_dca"],

        "spd_ratio": test_spd_summary["spd_ratio"],
        "improvement_pct": test_spd_summary["improvement_pct"],
        "result": get_status_from_pct_diff(test_spd_summary["improvement_pct"]),
    },
])

display(final_summary_df.round(6))


,period,windows,wins,losses,ties,win_rate_pct
0,train,6,5,1,0,83.33
1,test,2,2,0,0,100.00


,period,start_date,end_date,windows,wins,losses,ties,win_rate_pct,strategy_sats,dca_sats,extra_sats_vs_dca,strategy_spd_sum,dca_spd_sum,extra_spd_sum_vs_dca,strategy_spd_avg,dca_spd_avg,extra_spd_avg_vs_dca,spd_ratio,improvement_pct,result
0,train,2018-01-01,2023-12-31,6,5,1,0,83.333333,5.108030e+07,5.051275e+07,567545.897621,51080.300063,50512.754166,567.545898,8513.383344,8418.792361,94.590983,1.011236,1.123569,better
1,test,2024-01-01,2025-12-31,2,2,0,0,100.000000,2.691109e+06,2.584244e+06,106864.212380,2691.108610,2584.244397,106.864212,1345.554305,1292.122199,53.432106,1.041352,4.135221,better


In [53]:
# ============================================================
# Cell 11: Final presentation summary table
# ============================================================
# This cell creates a clean train/test summary table for presentation.
# SPD means Sats per Dollar. Higher SPD means more Bitcoin accumulated
# for each dollar invested.

train_spd_summary = summarize_spd_like_composite(train_window_summary_df)
test_spd_summary = summarize_spd_like_composite(test_window_summary_df)

final_summary_df = pd.DataFrame([
    {
        "Period": "Train",
        "Start Date": TRAIN_START,
        "End Date": TRAIN_END,
        "Windows": train_spd_summary["n_windows"],
        "Improvement (%)": train_spd_summary["improvement_pct"],
        "Win Rate (%)": train_spd_summary["win_rate_pct"],
        "Strategy Sats per Dollar": train_spd_summary["strategy_spd_sum"],
        "DCA Sats per Dollar": train_spd_summary["dca_spd_sum"],
        "Result": get_status_from_pct_diff(train_spd_summary["improvement_pct"]),
    },
    {
        "Period": "Test",
        "Start Date": TEST_START,
        "End Date": TEST_END,
        "Windows": test_spd_summary["n_windows"],
        "Improvement (%)": test_spd_summary["improvement_pct"],
        "Win Rate (%)": test_spd_summary["win_rate_pct"],
        "Strategy Sats per Dollar": test_spd_summary["strategy_spd_sum"],
        "DCA Sats per Dollar": test_spd_summary["dca_spd_sum"],
        "Result": get_status_from_pct_diff(test_spd_summary["improvement_pct"]),
    },
])

# Round numeric columns for cleaner presentation.
final_summary_df = final_summary_df.round({
    "Improvement (%)": 2,
    "Win Rate (%)": 2,
    "Strategy Sats per Dollar": 2,
    "DCA Sats per Dollar": 2,
})

display(final_summary_df)

,Period,Start Date,End Date,Windows,Improvement (%),Win Rate (%),Strategy Sats per Dollar,DCA Sats per Dollar,Result
0,Train,2018-01-01,2023-12-31,6,1.12,83.33,51080.30,50512.75,better
1,Test,2024-01-01,2025-12-31,2,4.14,100.00,2691.11,2584.24,better


In [54]:
# ============================================================
# Cell 12: Prepare plot data
# ============================================================
# This cell combines train and test daily outputs for plotting.

train_plot_df = train_strategy_df.copy()
train_plot_df["split"] = "Train"

test_plot_df = test_strategy_df.copy()
test_plot_df["split"] = "Test"

combined_strategy_df = pd.concat([train_plot_df, test_plot_df], ignore_index=True)
combined_strategy_df["date"] = pd.to_datetime(combined_strategy_df["date"])

train_plot_start = pd.to_datetime(TRAIN_START)
test_plot_start = pd.to_datetime(TEST_START)
test_plot_end = pd.to_datetime(TEST_END)


In [55]:
# ============================================================
# Cell 13: Main chart function
# ============================================================
# This cell defines the main BTC price vs strategy/DCA weight chart.
# It shows BTC price, final strategy weights, DCA weights, and train/test summary metrics.

def plot_train_test_period_with_spd_header(
    combined_df,
    train_summary,
    test_summary,
    title="BTC Price vs Strategy and Uniform DCA Weights",
):
    """
    Build the final train/test chart.

    Parameters
    ----------
    combined_df : pd.DataFrame
        Daily train and test strategy output combined into one dataframe.
    train_summary : dict
        Summary metrics for the training period.
    test_summary : dict
        Summary metrics for the test period.
    title : str
        Chart title.

    Returns
    -------
    plotly.graph_objects.Figure
        Plotly figure showing BTC price, strategy weight, DCA weight,
        and train/test performance text.
    """

    # Copy dataframe to avoid modifying original data.
    df = combined_df.copy()

    # Ensure date column is datetime.
    df["date"] = pd.to_datetime(df["date"])

    # Format train and test improvement annotations.
    train_arrow_text, train_arrow_color = format_arrow_text(
        train_summary["extra_spd_sum_vs_dca"],
        train_summary["improvement_pct"],
    )

    test_arrow_text, test_arrow_color = format_arrow_text(
        test_summary["extra_spd_sum_vs_dca"],
        test_summary["improvement_pct"],
    )

    # Build custom log-axis tick labels.
    tickvals, ticktext = build_log_tick_values_and_text()

    # Create annual x-axis ticks.
    year_ticks = pd.date_range(
        start=pd.to_datetime(TRAIN_START),
        end=pd.to_datetime(TEST_END),
        freq="YS",
    )

    # Create figure with a secondary y-axis.
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Add BTC price line on primary y-axis.
    fig.add_trace(
        go.Scatter(
            x=df["date"],
            y=df["price_usd"],
            mode="lines",
            name="BTC Price",
            line=dict(color="black", width=2.0),
            hovertemplate="<b>%{x|%Y-%m-%d}</b><br>Price: $%{y:,.0f}<extra></extra>",
        ),
        secondary_y=False,
    )

    # Add final strategy allocation weight on secondary y-axis.
    fig.add_trace(
        go.Scatter(
            x=df["date"],
            y=df["final_strategy_weight"],
            mode="lines",
            name="Strategy Weight",
            line=dict(color="green", width=1.8),
            hovertemplate="<b>%{x|%Y-%m-%d}</b><br>Strategy Weight: %{y:.6f}<extra></extra>",
        ),
        secondary_y=True,
    )

    # Add uniform DCA allocation line on secondary y-axis.
    fig.add_trace(
        go.Scatter(
            x=df["date"],
            y=df["dca_weight"],
            mode="lines",
            name="Uniform DCA (StackSats UniformStrategy)",
            line=dict(color="red", width=1.4, dash="dash"),
            hovertemplate="<b>%{x|%Y-%m-%d}</b><br>DCA Weight: %{y:.6f}<extra></extra>",
        ),
        secondary_y=True,
    )

    # Highlight test period with a light green rectangle.
    fig.add_vrect(
        x0=test_plot_start,
        x1=test_plot_end,
        fillcolor="lightgreen",
        opacity=0.15,
        layer="below",
        line_width=0,
    )

    # Add vertical line marking the train/test boundary.
    fig.add_vline(
        x=test_plot_start,
        line_width=1.2,
        line_dash="dot",
        line_color="gray",
    )

    # Add label for test region.
    fig.add_annotation(
        x=test_plot_start + pd.Timedelta(days=40),
        y=0.965,
        xref="x",
        yref="paper",
        text="<b>Test</b>",
        showarrow=False,
        font=dict(size=18, color="black", family="Arial Black"),
        xanchor="left",
    )

    # Configure BTC price axis.
    fig.update_yaxes(
        title_text="<b>BTC Price (USD, log scale)</b>",
        type="log",
        tickmode="array",
        tickvals=tickvals,
        ticktext=ticktext,
        secondary_y=False,
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        zeroline=False,
        title_font=dict(size=20, color="black", family="Arial Black"),
        tickfont=dict(size=13, color="black", family="Arial Black"),
    )

    # Configure allocation weight axis.
    fig.update_yaxes(
        title_text="<b>Allocation Weight</b>",
        secondary_y=True,
        showgrid=False,
        range=[0, 0.082],
        zeroline=False,
        title_font=dict(size=20, color="black", family="Arial Black"),
        tickfont=dict(size=13, color="black", family="Arial Black"),
    )

    # Configure x-axis.
    fig.update_xaxes(
        title_text="<b>Year</b>",
        range=[train_plot_start, test_plot_end],
        tickmode="array",
        tickvals=year_ticks,
        tickformat="%Y",
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        zeroline=False,
        title_font=dict(size=20, color="black", family="Arial Black"),
        tickfont=dict(size=14, color="black", family="Arial Black"),
    )

    # Configure overall chart layout.
    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b>",
            x=0.5,
            xanchor="center",
            y=0.985,
            font=dict(size=30, color="black", family="Arial Black"),
        ),
        template="plotly_white",
        hovermode="x unified",
        width=1900,
        height=1050,
        showlegend=False,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=35, r=35, t=120, b=35),
    )

    # Add train summary annotation above chart.
    fig.add_annotation(
        x=0.5,
        y=1.095,
        xref="paper",
        yref="paper",
        showarrow=False,
        align="center",
        text=(
            f"<b>Train — Strategy: {train_summary['strategy_spd_sum']:,.2f} sats/$ | "
            f"DCA: {train_summary['dca_spd_sum']:,.2f} sats/$ | "
            f"Excess: <span style='color:{train_arrow_color};'>{train_arrow_text}</span> | "
            f"Win Rate: {train_summary['win_rate_pct']:.1f}%</b>"
        ),
        font=dict(size=17, color="black", family="Arial Black"),
    )

    # Add test summary annotation above chart.
    fig.add_annotation(
        x=0.5,
        y=1.055,
        xref="paper",
        yref="paper",
        showarrow=False,
        align="center",
        text=(
            f"<b>Test — Strategy: {test_summary['strategy_spd_sum']:,.2f} sats/$ | "
            f"DCA: {test_summary['dca_spd_sum']:,.2f} sats/$ | "
            f"Excess: <span style='color:{test_arrow_color};'>{test_arrow_text}</span> | "
            f"Win Rate: {test_summary['win_rate_pct']:.1f}%</b>"
        ),
        font=dict(size=17, color="black", family="Arial Black"),
    )

    return fig


In [56]:
# ============================================================
# Cell 14: Show chart
# ============================================================
# This cell renders the final chart.

main_fig = plot_train_test_period_with_spd_header(
    combined_strategy_df,
    train_spd_summary,
    test_spd_summary,
)

main_fig.show()


# Tests/checks

In [57]:
# ============================================================
# Regime classification matrix: BTC regime x MVRV x Cap Growth
# ============================================================
# This cell shows how many days fall into each possible regime combination.
#
# Columns:
# - BTC Bull
# - BTC Bear
# - BTC Recovery
# - BTC Neutral
#
# Rows:
# - MVRV valuation regime
# - Cap-growth regime
#
# Values:
# - Count of days in each combination
#
# A value of 0 means that the combination is possible in theory,
# but did not appear in the selected train/test data.

import pandas as pd

# Choose which dataframe to analyze.
# Option 1: selected train + test daily strategy output
regime_source_df = combined_strategy_df.copy()

# If combined_strategy_df is not available yet, use:
# regime_source_df = pd.concat([train_strategy_df, test_strategy_df], ignore_index=True)

# ------------------------------------------------------------
# Split combined_regime into separate components
# Format is:
# BTC trend regime | MVRV valuation regime | cap-growth regime
# ------------------------------------------------------------
regime_parts = regime_source_df["combined_regime"].str.split(" | ", regex=False, expand=True)

regime_source_df["btc_regime"] = regime_parts[0]
regime_source_df["mvrv_regime"] = regime_parts[1]
regime_source_df["cap_growth_regime"] = regime_parts[2]

# ------------------------------------------------------------
# Define all expected categories
# ------------------------------------------------------------
btc_regime_order = [
    "BTC Bull",
    "BTC Bear",
    "BTC Recovery",
    "BTC Neutral",
]

mvrv_regime_order = [
    "Low MVRV",
    "Normal MVRV",
    "High MVRV",
]

cap_growth_order = [
    "Realized Growth Leading",
    "Market Growth Leading",
]

# ------------------------------------------------------------
# Create matrix table
# Rows = MVRV + Cap Growth
# Columns = BTC regime
# Values = day count
# ------------------------------------------------------------
regime_matrix_df = (
    regime_source_df
    .groupby(["mvrv_regime", "cap_growth_regime", "btc_regime"])
    .size()
    .reset_index(name="day_count")
    .pivot_table(
        index=["mvrv_regime", "cap_growth_regime"],
        columns="btc_regime",
        values="day_count",
        fill_value=0,
        aggfunc="sum",
    )
)

# Reindex so all possible combinations appear, even if count is 0
full_row_index = pd.MultiIndex.from_product(
    [mvrv_regime_order, cap_growth_order],
    names=["mvrv_regime", "cap_growth_regime"],
)

regime_matrix_df = (
    regime_matrix_df
    .reindex(index=full_row_index, columns=btc_regime_order, fill_value=0)
    .astype(int)
)

# Add row totals
regime_matrix_df["Total"] = regime_matrix_df.sum(axis=1)

# Add column totals
regime_matrix_with_total_df = regime_matrix_df.copy()
regime_matrix_with_total_df.loc[("Total", ""), :] = regime_matrix_with_total_df.sum(axis=0)

display(regime_matrix_with_total_df)

btc_regime                           BTC Bull  BTC Bear  BTC Recovery  \
mvrv_regime cap_growth_regime                                           
Low MVRV    Realized Growth Leading       0.0     314.0           1.0   
            Market Growth Leading         0.0       4.0           0.0   
Normal MVRV Realized Growth Leading     288.0     351.0          25.0   
            Market Growth Leading       854.0     110.0          47.0   
High MVRV   Realized Growth Leading       0.0       0.0           0.0   
            Market Growth Leading       279.0       0.0           0.0   
Total                                  1421.0     779.0          73.0   

btc_regime                           BTC Neutral   Total  
mvrv_regime cap_growth_regime                             
Low MVRV    Realized Growth Leading          0.0   315.0  
            Market Growth Leading            0.0     4.0  
Normal MVRV Realized Growth Leading        302.0   966.0  
            Market Growth Leading          340.0  1351.0  
High MVRV   Realized Growth Leading          0.0     0.0  
            Market Growth Leading            5.0   284.0  
Total                                      647.0  2920.0

In [58]:
# ============================================================
# Train/Test regime classification matrices
# ============================================================

def build_regime_matrix(input_df):
    df = input_df.copy()

    regime_parts = df["combined_regime"].str.split(" | ", regex=False, expand=True)

    df["btc_regime"] = regime_parts[0]
    df["mvrv_regime"] = regime_parts[1]
    df["cap_growth_regime"] = regime_parts[2]

    btc_regime_order = [
        "BTC Bull",
        "BTC Bear",
        "BTC Recovery",
        "BTC Neutral",
    ]

    mvrv_regime_order = [
        "Low MVRV",
        "Normal MVRV",
        "High MVRV",
    ]

    cap_growth_order = [
        "Realized Growth Leading",
        "Market Growth Leading",
    ]

    matrix_df = (
        df
        .groupby(["mvrv_regime", "cap_growth_regime", "btc_regime"])
        .size()
        .reset_index(name="day_count")
        .pivot_table(
            index=["mvrv_regime", "cap_growth_regime"],
            columns="btc_regime",
            values="day_count",
            fill_value=0,
            aggfunc="sum",
        )
    )

    full_row_index = pd.MultiIndex.from_product(
        [mvrv_regime_order, cap_growth_order],
        names=["mvrv_regime", "cap_growth_regime"],
    )

    matrix_df = (
        matrix_df
        .reindex(index=full_row_index, columns=btc_regime_order, fill_value=0)
        .astype(int)
    )

    matrix_df["Total"] = matrix_df.sum(axis=1)

    matrix_with_total_df = matrix_df.copy()
    matrix_with_total_df.loc[("Total", ""), :] = matrix_with_total_df.sum(axis=0)

    return matrix_with_total_df


train_regime_matrix_df = build_regime_matrix(train_strategy_df)
test_regime_matrix_df = build_regime_matrix(test_strategy_df)

print("Train regime classification matrix")
display(train_regime_matrix_df)

print("Test regime classification matrix")
display(test_regime_matrix_df)

Train regime classification matrix


btc_regime                           BTC Bull  BTC Bear  BTC Recovery  \
mvrv_regime cap_growth_regime                                           
Low MVRV    Realized Growth Leading       0.0     314.0           1.0   
            Market Growth Leading         0.0       4.0           0.0   
Normal MVRV Realized Growth Leading     254.0     305.0          24.0   
            Market Growth Leading       458.0     106.0          33.0   
High MVRV   Realized Growth Leading       0.0       0.0           0.0   
            Market Growth Leading       213.0       0.0           0.0   
Total                                   925.0     729.0          58.0   

btc_regime                           BTC Neutral   Total  
mvrv_regime cap_growth_regime                             
Low MVRV    Realized Growth Leading          0.0   315.0  
            Market Growth Leading            0.0     4.0  
Normal MVRV Realized Growth Leading        218.0   801.0  
            Market Growth Leading          255.0   852.0  
High MVRV   Realized Growth Leading          0.0     0.0  
            Market Growth Leading            5.0   218.0  
Total                                      478.0  2190.0

Test regime classification matrix


btc_regime                           BTC Bull  BTC Bear  BTC Recovery  \
mvrv_regime cap_growth_regime                                           
Low MVRV    Realized Growth Leading       0.0       0.0           0.0   
            Market Growth Leading         0.0       0.0           0.0   
Normal MVRV Realized Growth Leading      34.0      46.0           1.0   
            Market Growth Leading       396.0       4.0          14.0   
High MVRV   Realized Growth Leading       0.0       0.0           0.0   
            Market Growth Leading        66.0       0.0           0.0   
Total                                   496.0      50.0          15.0   

btc_regime                           BTC Neutral  Total  
mvrv_regime cap_growth_regime                            
Low MVRV    Realized Growth Leading          0.0    0.0  
            Market Growth Leading            0.0    0.0  
Normal MVRV Realized Growth Leading         84.0  165.0  
            Market Growth Leading           85.0  499.0  
High MVRV   Realized Growth Leading          0.0    0.0  
            Market Growth Leading            0.0   66.0  
Total                                      169.0  730.0

In [59]:
display(train_regime_matrix_df.astype(int))
display(test_regime_matrix_df.astype(int))

btc_regime                           BTC Bull  BTC Bear  BTC Recovery  \
mvrv_regime cap_growth_regime                                           
Low MVRV    Realized Growth Leading         0       314             1   
            Market Growth Leading           0         4             0   
Normal MVRV Realized Growth Leading       254       305            24   
            Market Growth Leading         458       106            33   
High MVRV   Realized Growth Leading         0         0             0   
            Market Growth Leading         213         0             0   
Total                                     925       729            58   

btc_regime                           BTC Neutral  Total  
mvrv_regime cap_growth_regime                            
Low MVRV    Realized Growth Leading            0    315  
            Market Growth Leading              0      4  
Normal MVRV Realized Growth Leading          218    801  
            Market Growth Leading            255    852  
High MVRV   Realized Growth Leading            0      0  
            Market Growth Leading              5    218  
Total                                        478   2190

btc_regime                           BTC Bull  BTC Bear  BTC Recovery  \
mvrv_regime cap_growth_regime                                           
Low MVRV    Realized Growth Leading         0         0             0   
            Market Growth Leading           0         0             0   
Normal MVRV Realized Growth Leading        34        46             1   
            Market Growth Leading         396         4            14   
High MVRV   Realized Growth Leading         0         0             0   
            Market Growth Leading          66         0             0   
Total                                     496        50            15   

btc_regime                           BTC Neutral  Total  
mvrv_regime cap_growth_regime                            
Low MVRV    Realized Growth Leading            0      0  
            Market Growth Leading              0      0  
Normal MVRV Realized Growth Leading           84    165  
            Market Growth Leading             85    499  
High MVRV   Realized Growth Leading            0      0  
            Market Growth Leading              0     66  
Total                                        169    730

In [60]:
# ============================================================
# Regime matrix with frequency classification
# ============================================================
# Labels:
# - Not observed: count = 0
# - Rare: 1 to 19 days
# - Possible / Seen: 20 to 99 days
# - Common: 100+ days

def classify_regime_count(day_count):
    if day_count == 0:
        return "Not observed"
    elif day_count < 20:
        return "Rare"
    elif day_count < 100:
        return "Possible / Seen"
    else:
        return "Common"


def build_regime_matrix_with_labels(input_df):
    df = input_df.copy()

    regime_parts = df["combined_regime"].str.split(" | ", regex=False, expand=True)

    df["btc_regime"] = regime_parts[0]
    df["mvrv_regime"] = regime_parts[1]
    df["cap_growth_regime"] = regime_parts[2]

    btc_regime_order = [
        "BTC Bull",
        "BTC Bear",
        "BTC Recovery",
        "BTC Neutral",
    ]

    mvrv_regime_order = [
        "Low MVRV",
        "Normal MVRV",
        "High MVRV",
    ]

    cap_growth_order = [
        "Realized Growth Leading",
        "Market Growth Leading",
    ]

    count_matrix_df = (
        df
        .groupby(["mvrv_regime", "cap_growth_regime", "btc_regime"])
        .size()
        .reset_index(name="day_count")
        .pivot_table(
            index=["mvrv_regime", "cap_growth_regime"],
            columns="btc_regime",
            values="day_count",
            fill_value=0,
            aggfunc="sum",
        )
    )

    full_row_index = pd.MultiIndex.from_product(
        [mvrv_regime_order, cap_growth_order],
        names=["mvrv_regime", "cap_growth_regime"],
    )

    count_matrix_df = (
        count_matrix_df
        .reindex(index=full_row_index, columns=btc_regime_order, fill_value=0)
        .astype(int)
    )

    # Create label matrix: "count | classification"
    label_matrix_df = count_matrix_df.copy().astype(str)

    for col in btc_regime_order:
        label_matrix_df[col] = count_matrix_df[col].apply(
            lambda x: f"{x} | {classify_regime_count(x)}"
        )

    # Add total count separately
    label_matrix_df["Total Days"] = count_matrix_df.sum(axis=1)

    return label_matrix_df


train_regime_label_matrix_df = build_regime_matrix_with_labels(train_strategy_df)
test_regime_label_matrix_df = build_regime_matrix_with_labels(test_strategy_df)

print("Train regime classification matrix with frequency labels")
display(train_regime_label_matrix_df)

print("Test regime classification matrix with frequency labels")
display(test_regime_label_matrix_df)

Train regime classification matrix with frequency labels


btc_regime                                   BTC Bull          BTC Bear  \
mvrv_regime cap_growth_regime                                             
Low MVRV    Realized Growth Leading  0 | Not observed      314 | Common   
            Market Growth Leading    0 | Not observed          4 | Rare   
Normal MVRV Realized Growth Leading      254 | Common      305 | Common   
            Market Growth Leading        458 | Common      106 | Common   
High MVRV   Realized Growth Leading  0 | Not observed  0 | Not observed   
            Market Growth Leading        213 | Common  0 | Not observed   

btc_regime                                   BTC Recovery       BTC Neutral  \
mvrv_regime cap_growth_regime                                                 
Low MVRV    Realized Growth Leading              1 | Rare  0 | Not observed   
            Market Growth Leading        0 | Not observed  0 | Not observed   
Normal MVRV Realized Growth Leading  24 | Possible / Seen      218 | Common   
            Market Growth Leading    33 | Possible / Seen      255 | Common   
High MVRV   Realized Growth Leading      0 | Not observed  0 | Not observed   
            Market Growth Leading        0 | Not observed          5 | Rare   

btc_regime                           Total Days  
mvrv_regime cap_growth_regime                    
Low MVRV    Realized Growth Leading         315  
            Market Growth Leading             4  
Normal MVRV Realized Growth Leading         801  
            Market Growth Leading           852  
High MVRV   Realized Growth Leading           0  
            Market Growth Leading           218

Test regime classification matrix with frequency labels


btc_regime                                       BTC Bull  \
mvrv_regime cap_growth_regime                               
Low MVRV    Realized Growth Leading      0 | Not observed   
            Market Growth Leading        0 | Not observed   
Normal MVRV Realized Growth Leading  34 | Possible / Seen   
            Market Growth Leading            396 | Common   
High MVRV   Realized Growth Leading      0 | Not observed   
            Market Growth Leading    66 | Possible / Seen   

btc_regime                                       BTC Bear      BTC Recovery  \
mvrv_regime cap_growth_regime                                                 
Low MVRV    Realized Growth Leading      0 | Not observed  0 | Not observed   
            Market Growth Leading        0 | Not observed  0 | Not observed   
Normal MVRV Realized Growth Leading  46 | Possible / Seen          1 | Rare   
            Market Growth Leading                4 | Rare         14 | Rare   
High MVRV   Realized Growth Leading      0 | Not observed  0 | Not observed   
            Market Growth Leading        0 | Not observed  0 | Not observed   

btc_regime                                    BTC Neutral  Total Days  
mvrv_regime cap_growth_regime                                          
Low MVRV    Realized Growth Leading      0 | Not observed           0  
            Market Growth Leading        0 | Not observed           0  
Normal MVRV Realized Growth Leading  84 | Possible / Seen         165  
            Market Growth Leading    85 | Possible / Seen         499  
High MVRV   Realized Growth Leading      0 | Not observed           0  
            Market Growth Leading        0 | Not observed          66

In [61]:
# ============================================================
# Regime matrix with count, frequency label, and rule condition
# ============================================================
# This table shows:
# 1. BTC regime condition
# 2. MVRV regime condition
# 3. Cap-growth regime condition
# 4. Count of days in each combination
# 5. Frequency label: Not observed / Rare / Possible / Common

def classify_regime_count(day_count):
    if day_count == 0:
        return "Not observed"
    elif day_count < 20:
        return "Rare"
    elif day_count < 100:
        return "Possible / Seen"
    else:
        return "Common"


BTC_REGIME_CONDITIONS = {
    "BTC Bull": "price > SMA and lookback return > 0",
    "BTC Bear": "price < SMA and lookback return < 0",
    "BTC Recovery": "price < SMA but short-term momentum improving",
    "BTC Neutral": "does not clearly fit Bull, Bear, or Recovery",
}

MVRV_REGIME_CONDITIONS = {
    "Low MVRV": "mvrv < 1.0",
    "Normal MVRV": "1.0 <= mvrv <= 2.5",
    "High MVRV": "mvrv > 2.5",
}

CAP_GROWTH_CONDITIONS = {
    "Realized Growth Leading": "realized_cap_growth_rate > market_cap_growth_rate",
    "Market Growth Leading": "market_cap_growth_rate >= realized_cap_growth_rate",
}


def build_regime_detail_table(input_df, period_name):
    df = input_df.copy()

    regime_parts = df["combined_regime"].str.split(" | ", regex=False, expand=True)

    df["btc_regime"] = regime_parts[0]
    df["mvrv_regime"] = regime_parts[1]
    df["cap_growth_regime"] = regime_parts[2]

    btc_regime_order = [
        "BTC Bull",
        "BTC Bear",
        "BTC Recovery",
        "BTC Neutral",
    ]

    mvrv_regime_order = [
        "Low MVRV",
        "Normal MVRV",
        "High MVRV",
    ]

    cap_growth_order = [
        "Realized Growth Leading",
        "Market Growth Leading",
    ]

    # Count observed combinations
    observed_counts_df = (
        df
        .groupby(["mvrv_regime", "cap_growth_regime", "btc_regime"])
        .size()
        .reset_index(name="day_count")
    )

    # Create all theoretical combinations
    all_combinations = []

    for mvrv_regime in mvrv_regime_order:
        for cap_growth_regime in cap_growth_order:
            for btc_regime in btc_regime_order:
                all_combinations.append(
                    {
                        "period": period_name,
                        "btc_regime": btc_regime,
                        "btc_regime_condition": BTC_REGIME_CONDITIONS[btc_regime],
                        "mvrv_regime": mvrv_regime,
                        "mvrv_condition": MVRV_REGIME_CONDITIONS[mvrv_regime],
                        "cap_growth_regime": cap_growth_regime,
                        "cap_growth_condition": CAP_GROWTH_CONDITIONS[cap_growth_regime],
                    }
                )

    all_combinations_df = pd.DataFrame(all_combinations)

    detail_df = (
        all_combinations_df
        .merge(
            observed_counts_df,
            on=["mvrv_regime", "cap_growth_regime", "btc_regime"],
            how="left",
        )
    )

    detail_df["day_count"] = detail_df["day_count"].fillna(0).astype(int)

    detail_df["frequency_label"] = detail_df["day_count"].apply(classify_regime_count)

    detail_df["regime_combination"] = (
        detail_df["btc_regime"]
        + " | "
        + detail_df["mvrv_regime"]
        + " | "
        + detail_df["cap_growth_regime"]
    )

    detail_df = detail_df[
        [
            "period",
            "regime_combination",
            "btc_regime",
            "btc_regime_condition",
            "mvrv_regime",
            "mvrv_condition",
            "cap_growth_regime",
            "cap_growth_condition",
            "day_count",
            "frequency_label",
        ]
    ]

    return detail_df


train_regime_detail_df = build_regime_detail_table(train_strategy_df, "Train")
test_regime_detail_df = build_regime_detail_table(test_strategy_df, "Test")

print("Train regime detail table")
display(train_regime_detail_df)

print("Test regime detail table")
display(test_regime_detail_df)

Train regime detail table


,period,regime_combination,btc_regime,btc_regime_condition,mvrv_regime,mvrv_condition,cap_growth_regime,cap_growth_condition,day_count,frequency_label
0,Train,BTC Bull | Low MVRV | Realized Growth Leading,BTC Bull,price > SMA and lookback return > 0,Low MVRV,mvrv < 1.0,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,0,Not observed
1,Train,BTC Bear | Low MVRV | Realized Growth Leading,BTC Bear,price < SMA and lookback return < 0,Low MVRV,mvrv < 1.0,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,314,Common
2,Train,BTC Recovery | Low MVRV | Realized Growth Leading,BTC Recovery,price < SMA but short-term momentum improving,Low MVRV,mvrv < 1.0,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,1,Rare
3,Train,BTC Neutral | Low MVRV | Realized Growth Leading,BTC Neutral,"does not clearly fit Bull, Bear, or Recovery",Low MVRV,mvrv < 1.0,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,0,Not observed
4,Train,BTC Bull | Low MVRV | Market Growth Leading,BTC Bull,price > SMA and lookback return > 0,Low MVRV,mvrv < 1.0,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,0,Not observed
5,Train,BTC Bear | Low MVRV | Market Growth Leading,BTC Bear,price < SMA and lookback return < 0,Low MVRV,mvrv < 1.0,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,4,Rare
6,Train,BTC Recovery | Low MVRV | Market Growth Leading,BTC Recovery,price < SMA but short-term momentum improving,Low MVRV,mvrv < 1.0,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,0,Not observed
7,Train,BTC Neutral | Low MVRV | Market Growth Leading,BTC Neutral,"does not clearly fit Bull, Bear, or Recovery",Low MVRV,mvrv < 1.0,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,0,Not observed
8,Train,BTC Bull | Normal MVRV | Realized Growth Leading,BTC Bull,price > SMA and lookback return > 0,Normal MVRV,1.0 <= mvrv <= 2.5,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,254,Common
9,Train,BTC Bear | Normal MVRV | Realized Growth Leading,BTC Bear,price < SMA and lookback return < 0,Normal MVRV,1.0 <= mvrv <= 2.5,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,305,Common


Test regime detail table


,period,regime_combination,btc_regime,btc_regime_condition,mvrv_regime,mvrv_condition,cap_growth_regime,cap_growth_condition,day_count,frequency_label
0,Test,BTC Bull | Low MVRV | Realized Growth Leading,BTC Bull,price > SMA and lookback return > 0,Low MVRV,mvrv < 1.0,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,0,Not observed
1,Test,BTC Bear | Low MVRV | Realized Growth Leading,BTC Bear,price < SMA and lookback return < 0,Low MVRV,mvrv < 1.0,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,0,Not observed
2,Test,BTC Recovery | Low MVRV | Realized Growth Leading,BTC Recovery,price < SMA but short-term momentum improving,Low MVRV,mvrv < 1.0,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,0,Not observed
3,Test,BTC Neutral | Low MVRV | Realized Growth Leading,BTC Neutral,"does not clearly fit Bull, Bear, or Recovery",Low MVRV,mvrv < 1.0,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,0,Not observed
4,Test,BTC Bull | Low MVRV | Market Growth Leading,BTC Bull,price > SMA and lookback return > 0,Low MVRV,mvrv < 1.0,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,0,Not observed
5,Test,BTC Bear | Low MVRV | Market Growth Leading,BTC Bear,price < SMA and lookback return < 0,Low MVRV,mvrv < 1.0,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,0,Not observed
6,Test,BTC Recovery | Low MVRV | Market Growth Leading,BTC Recovery,price < SMA but short-term momentum improving,Low MVRV,mvrv < 1.0,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,0,Not observed
7,Test,BTC Neutral | Low MVRV | Market Growth Leading,BTC Neutral,"does not clearly fit Bull, Bear, or Recovery",Low MVRV,mvrv < 1.0,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,0,Not observed
8,Test,BTC Bull | Normal MVRV | Realized Growth Leading,BTC Bull,price > SMA and lookback return > 0,Normal MVRV,1.0 <= mvrv <= 2.5,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,34,Possible / Seen
9,Test,BTC Bear | Normal MVRV | Realized Growth Leading,BTC Bear,price < SMA and lookback return < 0,Normal MVRV,1.0 <= mvrv <= 2.5,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,46,Possible / Seen


In [62]:
# ============================================================
# Regime matrix with frequency classification + rule conditions
# ============================================================
# Labels:
# - Not observed: count = 0
# - Rare: 1 to 19 days
# - Possible / Seen: 20 to 99 days
# - Common: 100+ days
#
# This table shows:
# - MVRV regime condition, for example Low MVRV = mvrv < 1.0
# - Cap-growth condition, for example Realized Growth Leading = realized_cap_growth_rate > market_cap_growth_rate
# - BTC regime columns with definitions
# - Count + frequency label for each regime combination

def classify_regime_count(day_count):
    if day_count == 0:
        return "Not observed"
    elif day_count < 20:
        return "Rare"
    elif day_count < 100:
        return "Possible / Seen"
    else:
        return "Common"


# ------------------------------------------------------------
# Rule definition dictionaries
# ------------------------------------------------------------
BTC_REGIME_CONDITIONS = {
    "BTC Bull": "price > SMA and lookback return > 0",
    "BTC Bear": "price < SMA and lookback return < 0",
    "BTC Recovery": "price < SMA but short-term momentum improving",
    "BTC Neutral": "does not clearly fit Bull, Bear, or Recovery",
}

MVRV_REGIME_CONDITIONS = {
    "Low MVRV": "mvrv < 1.0",
    "Normal MVRV": "1.0 <= mvrv <= 2.5",
    "High MVRV": "mvrv > 2.5",
}

CAP_GROWTH_CONDITIONS = {
    "Realized Growth Leading": "realized_cap_growth_rate > market_cap_growth_rate",
    "Market Growth Leading": "market_cap_growth_rate >= realized_cap_growth_rate",
}


def build_regime_matrix_with_labels(input_df):
    df = input_df.copy()

    regime_parts = df["combined_regime"].str.split(" | ", regex=False, expand=True)

    df["btc_regime"] = regime_parts[0]
    df["mvrv_regime"] = regime_parts[1]
    df["cap_growth_regime"] = regime_parts[2]

    btc_regime_order = [
        "BTC Bull",
        "BTC Bear",
        "BTC Recovery",
        "BTC Neutral",
    ]

    mvrv_regime_order = [
        "Low MVRV",
        "Normal MVRV",
        "High MVRV",
    ]

    cap_growth_order = [
        "Realized Growth Leading",
        "Market Growth Leading",
    ]

    count_matrix_df = (
        df
        .groupby(["mvrv_regime", "cap_growth_regime", "btc_regime"])
        .size()
        .reset_index(name="day_count")
        .pivot_table(
            index=["mvrv_regime", "cap_growth_regime"],
            columns="btc_regime",
            values="day_count",
            fill_value=0,
            aggfunc="sum",
        )
    )

    full_row_index = pd.MultiIndex.from_product(
        [mvrv_regime_order, cap_growth_order],
        names=["mvrv_regime", "cap_growth_regime"],
    )

    count_matrix_df = (
        count_matrix_df
        .reindex(index=full_row_index, columns=btc_regime_order, fill_value=0)
        .astype(int)
    )

    # ------------------------------------------------------------
    # Create label matrix: "count | classification"
    # ------------------------------------------------------------
    label_matrix_df = count_matrix_df.copy().astype(str)

    for col in btc_regime_order:
        label_matrix_df[col] = count_matrix_df[col].apply(
            lambda x: f"{x} | {classify_regime_count(x)}"
        )

    # ------------------------------------------------------------
    # Add total count
    # ------------------------------------------------------------
    label_matrix_df["Total Days"] = count_matrix_df.sum(axis=1)

    # ------------------------------------------------------------
    # Add condition columns for MVRV and cap growth
    # ------------------------------------------------------------
    label_matrix_df = label_matrix_df.reset_index()

    label_matrix_df["mvrv_condition"] = label_matrix_df["mvrv_regime"].map(
        MVRV_REGIME_CONDITIONS
    )

    label_matrix_df["cap_growth_condition"] = label_matrix_df["cap_growth_regime"].map(
        CAP_GROWTH_CONDITIONS
    )

    # ------------------------------------------------------------
    # Rename BTC columns so the condition is visible in same table
    # ------------------------------------------------------------
    rename_btc_columns = {
        btc_regime: f"{btc_regime} | {BTC_REGIME_CONDITIONS[btc_regime]}"
        for btc_regime in btc_regime_order
    }

    label_matrix_df = label_matrix_df.rename(columns=rename_btc_columns)

    # ------------------------------------------------------------
    # Reorder columns for readability
    # ------------------------------------------------------------
    final_btc_columns = list(rename_btc_columns.values())

    label_matrix_df = label_matrix_df[
        [
            "mvrv_regime",
            "mvrv_condition",
            "cap_growth_regime",
            "cap_growth_condition",
            *final_btc_columns,
            "Total Days",
        ]
    ]

    return label_matrix_df


train_regime_label_matrix_df = build_regime_matrix_with_labels(train_strategy_df)
test_regime_label_matrix_df = build_regime_matrix_with_labels(test_strategy_df)

print("Train regime classification matrix with frequency labels and rule conditions")
display(train_regime_label_matrix_df)

print("Test regime classification matrix with frequency labels and rule conditions")
display(test_regime_label_matrix_df)

Train regime classification matrix with frequency labels and rule conditions


btc_regime,mvrv_regime,mvrv_condition,cap_growth_regime,cap_growth_condition,BTC Bull | price > SMA and lookback return > 0,BTC Bear | price < SMA and lookback return < 0,BTC Recovery | price < SMA but short-term momentum improving,"BTC Neutral | does not clearly fit Bull, Bear, or Recovery",Total Days
0,Low MVRV,mvrv < 1.0,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,0 | Not observed,314 | Common,1 | Rare,0 | Not observed,315
1,Low MVRV,mvrv < 1.0,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,0 | Not observed,4 | Rare,0 | Not observed,0 | Not observed,4
2,Normal MVRV,1.0 <= mvrv <= 2.5,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,254 | Common,305 | Common,24 | Possible / Seen,218 | Common,801
3,Normal MVRV,1.0 <= mvrv <= 2.5,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,458 | Common,106 | Common,33 | Possible / Seen,255 | Common,852
4,High MVRV,mvrv > 2.5,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,0 | Not observed,0 | Not observed,0 | Not observed,0 | Not observed,0
5,High MVRV,mvrv > 2.5,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,213 | Common,0 | Not observed,0 | Not observed,5 | Rare,218


Test regime classification matrix with frequency labels and rule conditions


btc_regime,mvrv_regime,mvrv_condition,cap_growth_regime,cap_growth_condition,BTC Bull | price > SMA and lookback return > 0,BTC Bear | price < SMA and lookback return < 0,BTC Recovery | price < SMA but short-term momentum improving,"BTC Neutral | does not clearly fit Bull, Bear, or Recovery",Total Days
0,Low MVRV,mvrv < 1.0,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,0 | Not observed,0 | Not observed,0 | Not observed,0 | Not observed,0
1,Low MVRV,mvrv < 1.0,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,0 | Not observed,0 | Not observed,0 | Not observed,0 | Not observed,0
2,Normal MVRV,1.0 <= mvrv <= 2.5,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,34 | Possible / Seen,46 | Possible / Seen,1 | Rare,84 | Possible / Seen,165
3,Normal MVRV,1.0 <= mvrv <= 2.5,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,396 | Common,4 | Rare,14 | Rare,85 | Possible / Seen,499
4,High MVRV,mvrv > 2.5,Realized Growth Leading,realized_cap_growth_rate > market_cap_growth_rate,0 | Not observed,0 | Not observed,0 | Not observed,0 | Not observed,0
5,High MVRV,mvrv > 2.5,Market Growth Leading,market_cap_growth_rate >= realized_cap_growth_...,66 | Possible / Seen,0 | Not observed,0 | Not observed,0 | Not observed,66


In [63]:

train_regimes = set(best_mapping_df["combined_regime"])
test_regimes = set(test_strategy_df["combined_regime"])

unseen_test_regimes = sorted(test_regimes - train_regimes)

print("Number of unseen test regimes:", len(unseen_test_regimes))
print("Unseen test regimes:")
for regime in unseen_test_regimes:
    print(regime)

fallback_days = test_strategy_df[
    ~test_strategy_df["combined_regime"].isin(train_regimes)
]

print("Fallback days in test:", len(fallback_days))

display(
    fallback_days[
        ["date", "combined_regime", "selected_strategy"]
    ].head(20)
)

Number of unseen test regimes: 2
Unseen test regimes:
BTC Recovery | Normal MVRV | Market Growth Leading
BTC Recovery | Normal MVRV | Realized Growth Leading
Fallback days in test: 15


,date,combined_regime,selected_strategy
237,2024-08-26,BTC Recovery | Normal MVRV | Market Growth Lea...,stacksats_mvrv_weight
244,2024-09-02,BTC Recovery | Normal MVRV | Market Growth Lea...,stacksats_mvrv_weight
245,2024-09-03,BTC Recovery | Normal MVRV | Market Growth Lea...,stacksats_mvrv_weight
247,2024-09-05,BTC Recovery | Normal MVRV | Market Growth Lea...,stacksats_mvrv_weight
274,2024-10-02,BTC Recovery | Normal MVRV | Market Growth Lea...,stacksats_mvrv_weight
275,2024-10-03,BTC Recovery | Normal MVRV | Market Growth Lea...,stacksats_mvrv_weight
276,2024-10-04,BTC Recovery | Normal MVRV | Market Growth Lea...,stacksats_mvrv_weight
277,2024-10-05,BTC Recovery | Normal MVRV | Market Growth Lea...,stacksats_mvrv_weight
278,2024-10-06,BTC Recovery | Normal MVRV | Market Growth Lea...,stacksats_mvrv_weight
279,2024-10-07,BTC Recovery | Normal MVRV | Market Growth Lea...,stacksats_mvrv_weight


In [64]:
# ============================================================
# Regime matrix with frequency classification + inline short conditions
# ============================================================
# Labels:
# - Not observed: count = 0
# - Rare: 1 to 19 days
# - Possible / Seen: 20 to 99 days
# - Common: 100+ days
#
# Short forms:
# - RC growth = realized_cap_growth_rate
# - MC growth = market_cap_growth_rate

def classify_regime_count(day_count):
    if day_count == 0:
        return "Not observed"
    elif day_count < 20:
        return "Rare"
    elif day_count < 100:
        return "Possible / Seen"
    else:
        return "Common"


BTC_REGIME_CONDITIONS = {
    "BTC Bull": "price > SMA, return > 0",
    "BTC Bear": "price < SMA, return < 0",
    "BTC Recovery": "price < SMA, momentum improving",
    "BTC Neutral": "no clear Bull/Bear/Recovery",
}

MVRV_REGIME_CONDITIONS = {
    "Low MVRV": "mvrv < 1.0",
    "Normal MVRV": "1.0 <= mvrv <= 2.5",
    "High MVRV": "mvrv > 2.5",
}

CAP_GROWTH_CONDITIONS = {
    "Realized Growth Leading": "RC growth > MC growth",
    "Market Growth Leading": "MC growth >= RC growth",
}


def build_regime_matrix_with_labels(input_df):
    df = input_df.copy()

    regime_parts = df["combined_regime"].str.split(" | ", regex=False, expand=True)

    df["btc_regime"] = regime_parts[0]
    df["mvrv_regime"] = regime_parts[1]
    df["cap_growth_regime"] = regime_parts[2]

    btc_regime_order = [
        "BTC Bull",
        "BTC Bear",
        "BTC Recovery",
        "BTC Neutral",
    ]

    mvrv_regime_order = [
        "Low MVRV",
        "Normal MVRV",
        "High MVRV",
    ]

    cap_growth_order = [
        "Realized Growth Leading",
        "Market Growth Leading",
    ]

    count_matrix_df = (
        df
        .groupby(["mvrv_regime", "cap_growth_regime", "btc_regime"])
        .size()
        .reset_index(name="day_count")
        .pivot_table(
            index=["mvrv_regime", "cap_growth_regime"],
            columns="btc_regime",
            values="day_count",
            fill_value=0,
            aggfunc="sum",
        )
    )

    full_row_index = pd.MultiIndex.from_product(
        [mvrv_regime_order, cap_growth_order],
        names=["mvrv_regime", "cap_growth_regime"],
    )

    count_matrix_df = (
        count_matrix_df
        .reindex(index=full_row_index, columns=btc_regime_order, fill_value=0)
        .astype(int)
    )

    label_matrix_df = count_matrix_df.copy().astype(str)

    for col in btc_regime_order:
        label_matrix_df[col] = count_matrix_df[col].apply(
            lambda x: f"{x} | {classify_regime_count(x)}"
        )

    label_matrix_df["Total Days"] = count_matrix_df.sum(axis=1)

    label_matrix_df = label_matrix_df.reset_index()

    label_matrix_df.insert(
        0,
        "Regime Group",
        range(1, len(label_matrix_df) + 1)
    )

    label_matrix_df["mvrv_regime"] = label_matrix_df["mvrv_regime"].apply(
        lambda x: f"{x} ({MVRV_REGIME_CONDITIONS[x]})"
    )

    label_matrix_df["cap_growth_regime"] = label_matrix_df["cap_growth_regime"].apply(
        lambda x: f"{x} ({CAP_GROWTH_CONDITIONS[x]})"
    )

    label_matrix_df = label_matrix_df.rename(
        columns={
            "mvrv_regime": "MVRV Regime",
            "cap_growth_regime": "Cap Growth Regime",

            "BTC Bull": f"BTC Bull ({BTC_REGIME_CONDITIONS['BTC Bull']})",
            "BTC Bear": f"BTC Bear ({BTC_REGIME_CONDITIONS['BTC Bear']})",
            "BTC Recovery": f"BTC Recovery ({BTC_REGIME_CONDITIONS['BTC Recovery']})",
            "BTC Neutral": f"BTC Neutral ({BTC_REGIME_CONDITIONS['BTC Neutral']})",
        }
    )

    return label_matrix_df


train_regime_label_matrix_df = build_regime_matrix_with_labels(train_strategy_df)
test_regime_label_matrix_df = build_regime_matrix_with_labels(test_strategy_df)

print("Train regime classification matrix")
display(train_regime_label_matrix_df.style.hide(axis="index"))

print("Test regime classification matrix")
display(test_regime_label_matrix_df.style.hide(axis="index"))

Train regime classification matrix


Regime Group,MVRV Regime,Cap Growth Regime,"BTC Bull (price > SMA, return > 0)","BTC Bear (price < SMA, return < 0)","BTC Recovery (price < SMA, momentum improving)",BTC Neutral (no clear Bull/Bear/Recovery),Total Days
1,Low MVRV (mvrv < 1.0),Realized Growth Leading (RC growth > MC growth),0 | Not observed,314 | Common,1 | Rare,0 | Not observed,315
2,Low MVRV (mvrv < 1.0),Market Growth Leading (MC growth >= RC growth),0 | Not observed,4 | Rare,0 | Not observed,0 | Not observed,4
3,Normal MVRV (1.0 <= mvrv <= 2.5),Realized Growth Leading (RC growth > MC growth),254 | Common,305 | Common,24 | Possible / Seen,218 | Common,801
4,Normal MVRV (1.0 <= mvrv <= 2.5),Market Growth Leading (MC growth >= RC growth),458 | Common,106 | Common,33 | Possible / Seen,255 | Common,852
5,High MVRV (mvrv > 2.5),Realized Growth Leading (RC growth > MC growth),0 | Not observed,0 | Not observed,0 | Not observed,0 | Not observed,0
6,High MVRV (mvrv > 2.5),Market Growth Leading (MC growth >= RC growth),213 | Common,0 | Not observed,0 | Not observed,5 | Rare,218


Test regime classification matrix


Regime Group,MVRV Regime,Cap Growth Regime,"BTC Bull (price > SMA, return > 0)","BTC Bear (price < SMA, return < 0)","BTC Recovery (price < SMA, momentum improving)",BTC Neutral (no clear Bull/Bear/Recovery),Total Days
1,Low MVRV (mvrv < 1.0),Realized Growth Leading (RC growth > MC growth),0 | Not observed,0 | Not observed,0 | Not observed,0 | Not observed,0
2,Low MVRV (mvrv < 1.0),Market Growth Leading (MC growth >= RC growth),0 | Not observed,0 | Not observed,0 | Not observed,0 | Not observed,0
3,Normal MVRV (1.0 <= mvrv <= 2.5),Realized Growth Leading (RC growth > MC growth),34 | Possible / Seen,46 | Possible / Seen,1 | Rare,84 | Possible / Seen,165
4,Normal MVRV (1.0 <= mvrv <= 2.5),Market Growth Leading (MC growth >= RC growth),396 | Common,4 | Rare,14 | Rare,85 | Possible / Seen,499
5,High MVRV (mvrv > 2.5),Realized Growth Leading (RC growth > MC growth),0 | Not observed,0 | Not observed,0 | Not observed,0 | Not observed,0
6,High MVRV (mvrv > 2.5),Market Growth Leading (MC growth >= RC growth),66 | Possible / Seen,0 | Not observed,0 | Not observed,0 | Not observed,66


In [65]:
# ============================================================
# Regime matrix with frequency classification + grouped headers
# ============================================================

def classify_regime_count(day_count):
    if day_count == 0:
        return "Not observed"
    elif day_count < 20:
        return "Rare"
    elif day_count < 100:
        return "Possible / Seen"
    else:
        return "Common"


BTC_REGIME_CONDITIONS = {
    "BTC Bull": "price > SMA, return > 0",
    "BTC Bear": "price < SMA, return < 0",
    "BTC Recovery": "price < SMA, momentum improving",
    "BTC Neutral": "no clear Bull/Bear/Recovery",
}

MVRV_REGIME_CONDITIONS = {
    "Low MVRV": "mvrv < 1.0",
    "Normal MVRV": "1.0 <= mvrv <= 2.5",
    "High MVRV": "mvrv > 2.5",
}


def build_regime_matrix_with_labels(input_df):
    df = input_df.copy()

    regime_parts = df["combined_regime"].str.split(" | ", regex=False, expand=True)

    df["btc_regime"] = regime_parts[0]
    df["mvrv_regime"] = regime_parts[1]
    df["cap_growth_regime"] = regime_parts[2]

    btc_regime_order = [
        "BTC Bull",
        "BTC Bear",
        "BTC Recovery",
        "BTC Neutral",
    ]

    mvrv_regime_order = [
        "Low MVRV",
        "Normal MVRV",
        "High MVRV",
    ]

    cap_growth_order = [
        "Realized Growth Leading",
        "Market Growth Leading",
    ]

    count_matrix_df = (
        df
        .groupby(["mvrv_regime", "cap_growth_regime", "btc_regime"])
        .size()
        .reset_index(name="day_count")
        .pivot_table(
            index=["mvrv_regime", "cap_growth_regime"],
            columns="btc_regime",
            values="day_count",
            fill_value=0,
            aggfunc="sum",
        )
    )

    full_row_index = pd.MultiIndex.from_product(
        [mvrv_regime_order, cap_growth_order],
        names=["MVRV Regime", "Cap Growth Regime"],
    )

    count_matrix_df = (
        count_matrix_df
        .reindex(index=full_row_index, columns=btc_regime_order, fill_value=0)
        .astype(int)
    )

    label_matrix_df = count_matrix_df.copy().astype(str)

    for col in btc_regime_order:
        label_matrix_df[col] = count_matrix_df[col].apply(
            lambda x: f"{x} | {classify_regime_count(x)}"
        )

    label_matrix_df["Total Days"] = count_matrix_df.sum(axis=1)

    # Add short conditions only to MVRV labels.
    # Keep Cap Growth labels clean without RC/MC growth condition text.
    new_index = []
    for mvrv_regime, cap_growth_regime in label_matrix_df.index:
        new_index.append(
            (
                f"{mvrv_regime} ({MVRV_REGIME_CONDITIONS[mvrv_regime]})",
                cap_growth_regime,
            )
        )

    label_matrix_df.index = pd.MultiIndex.from_tuples(
        new_index,
        names=["MVRV Regime", "Cap Growth Regime"],
    )

    # Add short conditions to Bitcoin market trend column labels.
    renamed_columns = {
        "BTC Bull": f"BTC Bull ({BTC_REGIME_CONDITIONS['BTC Bull']})",
        "BTC Bear": f"BTC Bear ({BTC_REGIME_CONDITIONS['BTC Bear']})",
        "BTC Recovery": f"BTC Recovery ({BTC_REGIME_CONDITIONS['BTC Recovery']})",
        "BTC Neutral": f"BTC Neutral ({BTC_REGIME_CONDITIONS['BTC Neutral']})",
        "Total Days": "Total Days",
    }

    label_matrix_df = label_matrix_df.rename(columns=renamed_columns)

    # Create grouped column header.
    label_matrix_df.columns = pd.MultiIndex.from_tuples(
        [
            ("Bitcoin Market Trend", col)
            if col != "Total Days"
            else ("", "Total Days")
            for col in label_matrix_df.columns
        ]
    )

    return label_matrix_df


train_regime_label_matrix_df = build_regime_matrix_with_labels(train_strategy_df)
test_regime_label_matrix_df = build_regime_matrix_with_labels(test_strategy_df)

print("Train regime classification matrix")
display(train_regime_label_matrix_df)

print("Test regime classification matrix")
display(test_regime_label_matrix_df)

Train regime classification matrix


Bitcoin Market Trend  \
                                                         BTC Bull (price > SMA, return > 0)   
MVRV Regime                      Cap Growth Regime                                            
Low MVRV (mvrv < 1.0)            Realized Growth Leading                   0 | Not observed   
                                 Market Growth Leading                     0 | Not observed   
Normal MVRV (1.0 <= mvrv <= 2.5) Realized Growth Leading                       254 | Common   
                                 Market Growth Leading                         458 | Common   
High MVRV (mvrv > 2.5)           Realized Growth Leading                   0 | Not observed   
                                 Market Growth Leading                         213 | Common   

                                                                                             \
                                                         BTC Bear (price < SMA, return < 0)   
MVRV Regime                      Cap Growth Regime                                            
Low MVRV (mvrv < 1.0)            Realized Growth Leading                       314 | Common   
                                 Market Growth Leading                             4 | Rare   
Normal MVRV (1.0 <= mvrv <= 2.5) Realized Growth Leading                       305 | Common   
                                 Market Growth Leading                         106 | Common   
High MVRV (mvrv > 2.5)           Realized Growth Leading                   0 | Not observed   
                                 Market Growth Leading                     0 | Not observed   

                                                                                                         \
                                                         BTC Recovery (price < SMA, momentum improving)   
MVRV Regime                      Cap Growth Regime                                                        
Low MVRV (mvrv < 1.0)            Realized Growth Leading                                       1 | Rare   
                                 Market Growth Leading                                 0 | Not observed   
Normal MVRV (1.0 <= mvrv <= 2.5) Realized Growth Leading                           24 | Possible / Seen   
                                 Market Growth Leading                             33 | Possible / Seen   
High MVRV (mvrv > 2.5)           Realized Growth Leading                               0 | Not observed   
                                 Market Growth Leading                                 0 | Not observed   

                                                                                                    \
                                                         BTC Neutral (no clear Bull/Bear/Recovery)   
MVRV Regime                      Cap Growth Regime                                                   
Low MVRV (mvrv < 1.0)            Realized Growth Leading                          0 | Not observed   
                                 Market Growth Leading                            0 | Not observed   
Normal MVRV (1.0 <= mvrv <= 2.5) Realized Growth Leading                              218 | Common   
                                 Market Growth Leading                                255 | Common   
High MVRV (mvrv > 2.5)           Realized Growth Leading                          0 | Not observed   
                                 Market Growth Leading                                    5 | Rare   

                                                                     
                                                         Total Days  
MVRV Regime                      Cap Growth Regime                   
Low MVRV (mvrv < 1.0)            Realized Growth Leading        315  
                                 Market Growth Leading            4  
Normal MVRV (1.0 <= mvrv <= 2.5) Realized Growth Leading        801  
                                 Market Growth Leading  

Test regime classification matrix


Bitcoin Market Trend  \
                                                         BTC Bull (price > SMA, return > 0)   
MVRV Regime                      Cap Growth Regime                                            
Low MVRV (mvrv < 1.0)            Realized Growth Leading                   0 | Not observed   
                                 Market Growth Leading                     0 | Not observed   
Normal MVRV (1.0 <= mvrv <= 2.5) Realized Growth Leading               34 | Possible / Seen   
                                 Market Growth Leading                         396 | Common   
High MVRV (mvrv > 2.5)           Realized Growth Leading                   0 | Not observed   
                                 Market Growth Leading                 66 | Possible / Seen   

                                                                                             \
                                                         BTC Bear (price < SMA, return < 0)   
MVRV Regime                      Cap Growth Regime                                            
Low MVRV (mvrv < 1.0)            Realized Growth Leading                   0 | Not observed   
                                 Market Growth Leading                     0 | Not observed   
Normal MVRV (1.0 <= mvrv <= 2.5) Realized Growth Leading               46 | Possible / Seen   
                                 Market Growth Leading                             4 | Rare   
High MVRV (mvrv > 2.5)           Realized Growth Leading                   0 | Not observed   
                                 Market Growth Leading                     0 | Not observed   

                                                                                                         \
                                                         BTC Recovery (price < SMA, momentum improving)   
MVRV Regime                      Cap Growth Regime                                                        
Low MVRV (mvrv < 1.0)            Realized Growth Leading                               0 | Not observed   
                                 Market Growth Leading                                 0 | Not observed   
Normal MVRV (1.0 <= mvrv <= 2.5) Realized Growth Leading                                       1 | Rare   
                                 Market Growth Leading                                        14 | Rare   
High MVRV (mvrv > 2.5)           Realized Growth Leading                               0 | Not observed   
                                 Market Growth Leading                                 0 | Not observed   

                                                                                                    \
                                                         BTC Neutral (no clear Bull/Bear/Recovery)   
MVRV Regime                      Cap Growth Regime                                                   
Low MVRV (mvrv < 1.0)            Realized Growth Leading                          0 | Not observed   
                                 Market Growth Leading                            0 | Not observed   
Normal MVRV (1.0 <= mvrv <= 2.5) Realized Growth Leading                      84 | Possible / Seen   
                                 Market Growth Leading                        85 | Possible / Seen   
High MVRV (mvrv > 2.5)           Realized Growth Leading                          0 | Not observed   
                                 Market Growth Leading                            0 | Not observed   

                                                                     
                                                         Total Days  
MVRV Regime                      Cap Growth Regime                   
Low MVRV (mvrv < 1.0)            Realized Growth Leading          0  
                                 Market Growth Leading            0  
Normal MVRV (1.0 <= mvrv <= 2.5) Realized Growth Leading        165  
                                 Market Growth Leading  